# Longitudinal Aggregation Results

**Hypothesis:** Target–aggregation matching — state aggregations (mean, concat) should win for **Arm A** (absolute age); change aggregations (annualized_rate, difference, lme_slope_change) should win for **Arm B** (brain change rate = PC1 of annualized FS ROI feature change).

Two extractors: FS ROI (657 features) and all CNN variants (120 files: 10 seeds × 4 channels × 3 scalers × 2 architectures).  
Metric: MAE, 5-fold nested GroupKFold CV.


In [ ]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, json
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors
import seaborn as sns
from scipy import stats
from scipy.stats import wilcoxon
from pathlib import Path
    
# Resolve project root regardless of whether Jupyter CWD is the
# project root or the brainage_agg/ subdirectory.
_cwd = Path().resolve()
# If outputs/results.csv is reachable, CWD is brainage_agg/ — step up one level.
if   (_cwd        / 'brainage_agg' / 'outputs' / 'results.csv').exists():
    root_dir = _cwd
elif (_cwd.parent / 'brainage_agg' / 'outputs' / 'results.csv').exists():
    root_dir = _cwd.parent
else:
    raise FileNotFoundError(
        f'Cannot locate brainage_agg/outputs/results.csv from {_cwd}. '
        'Run Jupyter from the project root or brainage_agg/.')

sns.set_theme(style='whitegrid', font_scale=1.1)

PALETTE       = {'fs_roi': '#2196F3', 'cnn': '#FF9800'}
AGG_ORDER_A   = ['mean', 'concatenation', 'annualized_rate', 'lme_slope']
AGG_ORDER_B   = ['annualized_rate', 'lme_slope_change', 'difference', 'mean', 'concatenation']
AGG_LABELS    = {
    'mean': 'Mean', 'concatenation': 'Concat',
    'annualized_rate': 'Ann. Rate', 'lme_slope': 'LME Slope',
    'difference': 'Difference', 'lme_slope_change': 'LME Slope\n(change)',
}
ARCH_COLORS   = {'cov_pool': '#5C6BC0', 'double_conv': '#EF6C00'}
SCALER_COLORS = {'minmax': '#EF5350', 'zscore': '#FFA726', 'robust': '#66BB6A'}
CHANNEL_COLORS= {'t1': '#78909C', 't1_sobel': '#42A5F5',
                 't1_rank_sobel': '#26A69A', 't1_median_sobel': '#AB47BC'}

df = pd.read_csv(root_dir / 'brainage_agg/outputs/results.csv')

# Per-file average over CV folds (one data point per extractor file × agg)
cnn_per_file = (
    df[(df['extractor_type']=='cnn') & (df['age_band']=='all')]
    .groupby(['cnn_arch','scaler','channels','cnn_seed','aggregation','target_arm'])['MAE']
    .mean().reset_index()
)

print(f'{len(df):,} rows total | CNN variants: {df[df["extractor_type"]=="cnn"]["extractor"].nunique()}')
print('Scaler × arch × channels combinations:')
print(cnn_per_file[['cnn_arch','scaler','channels']].drop_duplicates().groupby(['cnn_arch','scaler']).size().unstack())

## 1  Cohort overview

In [ ]:
manifest = pd.read_csv(root_dir / 'brainage_agg/outputs/manifest.csv')
manifest['ages']     = manifest['ages'].apply(json.loads)
manifest['age_first']= manifest['ages'].apply(lambda x: x[0])
manifest['age_last'] = manifest['ages'].apply(lambda x: x[-1])
manifest['delta_t']  = manifest['age_last'] - manifest['age_first']
long_df = manifest[manifest['cohort_all']].copy()

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for band, c in [('Child', '#4CAF50'), ('Adult', '#2196F3')]:
    s = long_df[long_df['band']==band]
    axes[0].hist(s['age_first'], bins=20, alpha=0.6, color=c, label=f'{band} (n={len(s)})')
axes[0].set(xlabel='Age at first scan (years)', ylabel='Count', title='Baseline age distribution')
axes[0].legend()

for band, c in [('Child', '#4CAF50'), ('Adult', '#2196F3')]:
    s = long_df[(long_df['band']==band) & (~long_df['exclude_arm_b'])]
    axes[1].hist(s['delta_t'], bins=15, alpha=0.6, color=c, label=f'{band} (n={len(s)})')
axes[1].set(xlabel='Δt = age_last − age_first (years)', ylabel='Count', title='Scan interval (Δt) distribution')
axes[1].legend()

tp_counts = long_df['n_timepoints'].value_counts().sort_index()
axes[2].bar(tp_counts.index, tp_counts.values, color='#9C27B0', alpha=0.8)
axes[2].set(xlabel='Timepoints per subject', ylabel='Subjects',
            title=f'Timepoints (cohort_all n={len(long_df)}, concat n={long_df["cohort_concat"].sum()})')
for x, y in zip(tp_counts.index, tp_counts.values):
    axes[2].text(x, y+1, str(y), ha='center', fontsize=10)

plt.tight_layout()
plt.savefig(root_dir / 'brainage_agg/outputs/figures/cohort_overview.png', dpi=150, bbox_inches='tight')
plt.show()

## 2  Headline heatmaps — target × aggregation matching

In [ ]:
def make_heatmap(ax, arm, agg_order, title):
    sub = df[(df['target_arm']==arm) & (df['age_band']=='all')].copy()
    sub['ext_label'] = sub['extractor_type'].map({'fs_roi':'FS ROI','cnn':'CNN (avg)'})
    cell_mean = sub.groupby(['aggregation','ext_label'])['MAE'].mean().reset_index()
    pivot = cell_mean.pivot(index='aggregation', columns='ext_label', values='MAE')
    pivot = pivot.reindex([a for a in agg_order if a in pivot.index])
    pivot.index = [AGG_LABELS.get(a, a) for a in pivot.index]
    sns.heatmap(pivot, annot=True, fmt='.3f', cmap='RdYlGn_r', ax=ax,
                vmin=pivot.values.min()*0.95, vmax=pivot.values.max()*1.05,
                linewidths=0.5, linecolor='white',
                cbar_kws={'label':'MAE (yrs)','shrink':0.8}, annot_kws={'size':12,'weight':'bold'})
    ax.set_title(title, fontsize=12, fontweight='bold', pad=8)
    ax.set(xlabel='', ylabel='')
    ax.tick_params(axis='both', rotation=0)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
make_heatmap(axes[0], 'A', AGG_ORDER_A, 'Arm A — Absolute age\n(state aggs expected to win)')
make_heatmap(axes[1], 'B', AGG_ORDER_B, 'Arm B — Brain change rate\n(change aggs expected to win)')
plt.suptitle('Target–Aggregation Matching Heatmap  (CNN column = mean over all 120 variants)',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(root_dir / 'brainage_agg/outputs/figures/matching_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

## 3  Arm A — state vs change aggregations

In [ ]:
arm_a = df[(df['target_arm']=='A') & (df['age_band']=='all')].copy()

# Best CNN config: arch × scaler × channels with lowest mean MAE (across seeds and folds)
cnn_a = arm_a[arm_a['extractor_type'] == 'cnn'].copy()
config_mae = (
    cnn_a.groupby(['cnn_arch','scaler','channels','aggregation'])['MAE']
    .mean().reset_index()
)
# Pick the config (arch×scaler×channels) with the lowest overall mean MAE across aggregations
config_overall = config_mae.groupby(['cnn_arch','scaler','channels'])['MAE'].mean()
best_cfg = config_overall.idxmin()  # (arch, scaler, channels)
best_arch, best_scaler, best_channels = best_cfg
print(f'Best CNN config (Arm A): arch={best_arch}  scaler={best_scaler}  channels={best_channels}')

best_cnn_a = arm_a[
    (arm_a['extractor_type']=='cnn') &
    (arm_a['cnn_arch']==best_arch) &
    (arm_a['scaler']==best_scaler) &
    (arm_a['channels']==best_channels)
].copy()
best_cnn_a['ext_label'] = f'CNN (best: {best_scaler})'

fs_a = arm_a[arm_a['extractor_type']=='fs_roi'].copy()
fs_a['ext_label'] = 'FS ROI'

plot_data = [
    ('FS ROI',                    fs_a,       '#1976D2'),
    (f'CNN (best: {best_scaler})', best_cnn_a, '#F57C00'),
]

fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=False)
for ax, (label, sub, c) in zip(axes, plot_data):
    by_ext = sub.groupby(['aggregation','extractor','fold'])['MAE'].mean().reset_index()
    by_agg = by_ext.groupby('aggregation')['MAE'].agg(['mean','std']).reindex(AGG_ORDER_A).reset_index()
    by_agg['label'] = by_agg['aggregation'].map(AGG_LABELS)
    colors = ['#43A047' if a in ['mean','concatenation'] else '#E53935' for a in by_agg['aggregation']]
    bars   = ax.bar(by_agg['label'], by_agg['mean'], color=colors, alpha=0.85,
                    yerr=by_agg['std'], capsize=4, error_kw={'lw':1.5})
    ax.set(ylabel='MAE (years)', title=f'Arm A — {label}')
    for bar, v in zip(bars, by_agg['mean']):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3, f'{v:.2f}',
                ha='center', va='bottom', fontsize=9, fontweight='bold')

axes[1].legend(handles=[
    mpatches.Patch(color='#43A047', label='State (expected to win)'),
    mpatches.Patch(color='#E53935', label='Change (expected to lose)')], fontsize=9)
plt.suptitle(
    f'Arm A: Absolute Age — MAE by Aggregation\n'
    f'CNN = best config (arch={best_arch}, scaler={best_scaler}, channels={best_channels})',
    fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(root_dir / 'brainage_agg/outputs/figures/arm_a_comparison.png', dpi=150, bbox_inches='tight')
plt.show()


## 4  Arm B — change descriptors vs negative controls


In [ ]:
arm_b = df[(df['target_arm']=='B') & (df['age_band']=='all')].copy()
chance_mae = arm_b['chance_mae'].mean()

# Best CNN config: arch × scaler × channels with lowest mean MAE across aggregations and seeds
cnn_b = arm_b[arm_b['extractor_type'] == 'cnn'].copy()
config_mae = (
    cnn_b.groupby(['cnn_arch','scaler','channels','aggregation'])['MAE']
    .mean().reset_index()
)
config_overall = config_mae.groupby(['cnn_arch','scaler','channels'])['MAE'].mean()
best_cfg = config_overall.idxmin()
best_arch, best_scaler, best_channels = best_cfg
print(f'Best CNN config (Arm B): arch={best_arch}  scaler={best_scaler}  channels={best_channels}')

best_cnn_b = arm_b[
    (arm_b['extractor_type']=='cnn') &
    (arm_b['cnn_arch']==best_arch) &
    (arm_b['scaler']==best_scaler) &
    (arm_b['channels']==best_channels)
].copy()

fs_b = arm_b[arm_b['extractor_type']=='fs_roi'].copy()

plot_data = [
    ('FS ROI',                     fs_b,       '#1976D2'),
    (f'CNN (best: {best_scaler})',  best_cnn_b, '#F57C00'),
]

fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=True)
for ax, (label, sub, c) in zip(axes, plot_data):
    by_ext = sub.groupby(['aggregation','extractor','fold'])['MAE'].mean().reset_index()
    by_agg = by_ext.groupby('aggregation')['MAE'].agg(['mean','std']).reindex(AGG_ORDER_B).reset_index()
    by_agg['label'] = by_agg['aggregation'].map(AGG_LABELS)
    is_ch  = by_agg['aggregation'].isin(['annualized_rate','difference','lme_slope_change'])
    bars   = ax.bar(by_agg['label'], by_agg['mean'],
                    color=['#1565C0' if x else '#B0BEC5' for x in is_ch],
                    alpha=0.85, yerr=by_agg['std'], capsize=4, error_kw={'lw':1.5})
    ax.axhline(chance_mae, color='red', linestyle='--', lw=1.5,
               label=f'Null = {chance_mae:.3f}')
    ax.set(ylabel='MAE', title=f'Arm B — {label}')
    ax.legend(fontsize=9)
    for bar, v in zip(bars, by_agg['mean']):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005, f'{v:.3f}',
                ha='center', va='bottom', fontsize=9, fontweight='bold')

plt.suptitle(
    f'Arm B: Brain Change Rate — MAE by Aggregation\n'
    f'CNN = best config (arch={best_arch}, scaler={best_scaler}, channels={best_channels})',
    fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(root_dir / 'brainage_agg/outputs/figures/arm_b_comparison.png', dpi=150, bbox_inches='tight')
plt.show()


## 4b  Arm B2 — eTIV rate target

**eTIV rate** = (eTIV_last − eTIV_first) / delta_t  [mm³/year] — a proxy brain-change target that avoids the PC1 leakage risk of Arm B (eTIV is not a direct input to FS ROI features).

⚠️ **Residual leakage warning (FS ROI):** eTIV is computed from FreeSurfer aseg.stats, which is correlated with FS ROI features. CNN features are free of this leakage. Interpret FS ROI B2 results with caution.

In [ ]:
# Arm B2 — eTIV rate: handle leakage_warning column gracefully
if 'leakage_warning' not in df.columns:
    df['leakage_warning'] = False
df['leakage_warning'] = df['leakage_warning'].fillna(False)

arm_b2 = df[(df['target_arm']=='B2') & (df['age_band']=='all')].copy()
if arm_b2.empty:
    print('Arm B2 results not yet available — rerun after NKI job completes.')
else:
    chance_mae_b2 = arm_b2['chance_mae'].mean()

    # Best CNN config: arch × scaler × channels with lowest mean MAE across aggregations and seeds
    cnn_b2 = arm_b2[arm_b2['extractor_type'] == 'cnn'].copy()
    config_mae = (
        cnn_b2.groupby(['cnn_arch','scaler','channels','aggregation'])['MAE']
        .mean().reset_index()
    )
    config_overall = config_mae.groupby(['cnn_arch','scaler','channels'])['MAE'].mean()
    best_cfg = config_overall.idxmin()
    best_arch, best_scaler, best_channels = best_cfg
    print(f'Best CNN config (Arm B2): arch={best_arch}  scaler={best_scaler}  channels={best_channels}')

    best_cnn_b2 = arm_b2[
        (arm_b2['extractor_type']=='cnn') &
        (arm_b2['cnn_arch']==best_arch) &
        (arm_b2['scaler']==best_scaler) &
        (arm_b2['channels']==best_channels)
    ].copy()

    fs_b2 = arm_b2[arm_b2['extractor_type']=='fs_roi'].copy()

    plot_data = [
        ('FS ROI',                     fs_b2,       '#1976D2'),
        (f'CNN (best: {best_scaler})',  best_cnn_b2, '#F57C00'),
    ]

    # ── Fig 1: Bar plot ─────────────────────────────────────────────────────────
    fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=False)
    for ax, (label, sub, c) in zip(axes, plot_data):
        by_ext = sub.groupby(['aggregation','extractor','fold'])['MAE'].mean().reset_index()
        by_agg = by_ext.groupby('aggregation')['MAE'].agg(['mean','std']).reindex(AGG_ORDER_B).reset_index()
        by_agg['label'] = by_agg['aggregation'].map(AGG_LABELS)
        is_ch  = by_agg['aggregation'].isin(['annualized_rate','difference','lme_slope_change'])
        bars   = ax.bar(by_agg['label'], by_agg['mean'],
                        color=['#1565C0' if x else '#B0BEC5' for x in is_ch],
                        alpha=0.85, yerr=by_agg['std'], capsize=4, error_kw={'lw':1.5})
        ax.axhline(chance_mae_b2, color='red', linestyle='--', lw=1.5,
                   label=f'Null = {chance_mae_b2:.0f} mm³/yr')
        ax.set(ylabel='MAE (mm³/year)', title=f'Arm B2 — {label}')
        ax.legend(fontsize=8)
        for bar, v in zip(bars, by_agg['mean']):
            ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.1*bar.get_height(),
                    f'{v:.0f}', ha='center', va='bottom', fontsize=9, fontweight='bold')

    # Annotate FS ROI leakage warning
    axes[0].annotate('⚠ Residual FS leakage\n(eTIV ∈ FS features)',
                     xy=(0.98, 0.97), xycoords='axes fraction',
                     ha='right', va='top', fontsize=8, color='#B71C1C',
                     bbox=dict(boxstyle='round,pad=0.3', fc='#FFEBEE', ec='#B71C1C', alpha=0.8))

    plt.suptitle(
        f'Arm B2: eTIV Rate — MAE by Aggregation\n'
        f'CNN = best config (arch={best_arch}, scaler={best_scaler}, channels={best_channels})',
        fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.savefig(root_dir / 'brainage_agg/outputs/figures/arm_b2_comparison.png', dpi=150, bbox_inches='tight')
    plt.show()

    # ── Fig 2: Heatmap (FS ROI vs best CNN) ─────────────────────────────────────
    fig, ax = plt.subplots(figsize=(6, 5))
    hm_data = pd.concat([
        fs_b2.assign(ext_label='FS ROI'),
        best_cnn_b2.assign(ext_label=f'CNN (best: {best_scaler})')
    ])
    cell_mean = hm_data.groupby(['aggregation','ext_label'])['MAE'].mean().reset_index()
    pivot = cell_mean.pivot(index='aggregation', columns='ext_label', values='MAE')
    pivot = pivot.reindex([a for a in AGG_ORDER_B if a in pivot.index])
    pivot.index = [AGG_LABELS.get(a, a) for a in pivot.index]
    sns.heatmap(pivot, annot=True, fmt='.0f', cmap='RdYlGn_r', ax=ax,
                linewidths=0.5, linecolor='white',
                cbar_kws={'label':'MAE (mm³/yr)','shrink':0.8}, annot_kws={'size':12,'weight':'bold'})
    ax.set_title('Arm B2 — eTIV Rate  (⚠ FS ROI has residual leakage)', fontsize=11, fontweight='bold', pad=8)
    ax.set(xlabel='', ylabel='')
    ax.tick_params(axis='both', rotation=0)
    plt.tight_layout()
    plt.savefig(root_dir / 'brainage_agg/outputs/figures/heatmap_arm_B2.png', dpi=150, bbox_inches='tight')
    plt.show()

    # ── Negative control check ───────────────────────────────────────────────────
    print('--- Negative control check (mean/concat vs chance) ---')
    for agg in ['mean', 'concatenation']:
        sub_nc = arm_b2[arm_b2['aggregation'] == agg]
        if not sub_nc.empty:
            mae_nc = sub_nc['MAE'].mean()
            ratio  = mae_nc / chance_mae_b2
            status = '✓ near chance' if ratio < 1.15 else '✗ suspiciously low'
            print(f'  {agg}: MAE={mae_nc:.0f}  chance={chance_mae_b2:.0f}  ratio={ratio:.2f}  {status}')

    # ── B vs B2 scatter — best CNN only ─────────────────────────────────────────
    cnn_b_best  = df[
        (df['target_arm']=='B') & (df['age_band']=='all') &
        (df['extractor_type']=='cnn') &
        (df['cnn_arch']==best_arch) & (df['scaler']==best_scaler) & (df['channels']==best_channels)
    ]
    if not cnn_b_best.empty and not best_cnn_b2.empty:
        b_agg  = cnn_b_best.groupby(['extractor','aggregation'])['MAE'].mean().reset_index().rename(columns={'MAE':'MAE_B'})
        b2_agg = best_cnn_b2.groupby(['extractor','aggregation'])['MAE'].mean().reset_index().rename(columns={'MAE':'MAE_B2'})
        merged = b_agg.merge(b2_agg, on=['extractor','aggregation'])
        if not merged.empty:
            merged['MAE_B_z']  = (merged['MAE_B']  - merged['MAE_B'].mean())  / merged['MAE_B'].std()
            merged['MAE_B2_z'] = (merged['MAE_B2'] - merged['MAE_B2'].mean()) / merged['MAE_B2'].std()
            fig, ax = plt.subplots(figsize=(7, 5))
            agg_colors = {'annualized_rate':'#E53935','lme_slope_change':'#FF7043',
                          'difference':'#8E24AA','mean':'#78909C','concatenation':'#455A64'}
            for agg_name, sub_m in merged.groupby('aggregation'):
                ax.scatter(sub_m['MAE_B_z'], sub_m['MAE_B2_z'],
                           color=agg_colors.get(agg_name, '#9E9E9E'),
                           alpha=0.6, s=25, label=AGG_LABELS.get(agg_name, agg_name))
            lims = [min(ax.get_xlim()[0], ax.get_ylim()[0]),
                    max(ax.get_xlim()[1], ax.get_ylim()[1])]
            ax.plot(lims, lims, 'k--', alpha=0.4, lw=1.5)
            r = merged['MAE_B_z'].corr(merged['MAE_B2_z'])
            ax.text(0.03, 0.97, f'r = {r:.2f}', transform=ax.transAxes,
                    va='top', fontsize=11, fontweight='bold')
            ax.set(xlabel='Arm B z-scored MAE', ylabel='Arm B2 z-scored MAE',
                   title=f'B vs B2 consistency (best CNN: {best_arch}/{best_scaler}/{best_channels})')
            ax.legend(fontsize=8, loc='lower right')
            plt.tight_layout()
            plt.savefig(root_dir / 'brainage_agg/outputs/figures/b_vs_b2_scatter.png', dpi=150, bbox_inches='tight')
            plt.show()
            print(f'B vs B2 Pearson r (z-scored MAE, best CNN): {r:.3f}')


## 5  Age band analysis (Child vs Adult)

In [ ]:
from scipy import stats as scipy_stats

BAND_COLORS = {'Child': '#43A047', 'Adult': '#1976D2'}
BANDS = ['Child', 'Adult']

def _band_stats(arm, ext_type, agg_order):
    """Mean ± 95% CI over folds.
    For CNN: average over seeds within each fold first, then std over folds.
    Makes error bars directly comparable between FS ROI and CNN (both N=5 folds).
    """
    sub = df[
        (df['target_arm'] == arm) &
        (df['extractor_type'] == ext_type) &
        (df['age_band'].isin(BANDS)) &
        (df['aggregation'].isin(agg_order))
    ].copy()
    # Average over seeds (and all CNN variants) within each fold → 1 value per fold
    by_fold = sub.groupby(['aggregation', 'age_band', 'fold'])['MAE'].mean().reset_index()
    n_folds = by_fold['fold'].nunique()
    t_crit  = scipy_stats.t.ppf(0.975, df=max(n_folds - 1, 1))
    stats_df = (by_fold.groupby(['aggregation', 'age_band'])['MAE']
                .agg(['mean', 'std', 'count'])
                .reset_index())
    stats_df['ci95'] = t_crit * stats_df['std'] / np.sqrt(stats_df['count'].clip(lower=1))
    return stats_df.set_index(['aggregation', 'age_band'])

def _draw_band_bars(ax, arm, ext_type, agg_order, ylabel=True):
    stats_df = _band_stats(arm, ext_type, agg_order)
    x, w = np.arange(len(agg_order)), 0.35
    for i, band in enumerate(BANDS):
        means = [stats_df.loc[(a, band), 'mean'] if (a, band) in stats_df.index else np.nan
                 for a in agg_order]
        ci95s = [stats_df.loc[(a, band), 'ci95']  if (a, band) in stats_df.index else 0
                 for a in agg_order]
        bars = ax.bar(x + i*w, means, w,
                      label=band, color=BAND_COLORS[band], alpha=0.85,
                      yerr=ci95s, capsize=4, error_kw={'lw': 1.5, 'capthick': 1.5})
        for bar, m, ci in zip(bars, means, ci95s):
            if not np.isnan(m):
                ax.text(bar.get_x() + bar.get_width() / 2,
                        m + ci + ax.get_ylim()[1] * 0.01,
                        f'{m:.2f}', ha='center', va='bottom', fontsize=7)
    ax.set_xticks(x + w / 2)
    ax.set_xticklabels([AGG_LABELS.get(a, a) for a in agg_order], fontsize=9)
    if ylabel:
        ax.set_ylabel('MAE (years)')
    ax.legend(fontsize=9)

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
for row, (arm, agg_order, arm_label) in enumerate([
    ('A', AGG_ORDER_A, 'Arm A — Absolute Age'),
    ('B', AGG_ORDER_B, 'Arm B — Brain Change Rate'),
]):
    for col, (ext_type, ext_label) in enumerate([
        ('fs_roi', 'FS ROI'),
        ('cnn',    'CNN (mean over variants)'),
    ]):
        ax = axes[row][col]
        _draw_band_bars(ax, arm, ext_type, agg_order, ylabel=(col == 0))
        ax.set_title(f'{arm_label} — {ext_label}', fontweight='bold')

axes[1][0].set_ylim(0, 4.5)

axes[0][1].sharey(axes[0][0])
axes[1][1].sharey(axes[1][0])
plt.setp(axes[0][1].get_yticklabels(), visible=False)
plt.setp(axes[1][1].get_yticklabels(), visible=False)

plt.suptitle(
    'MAE by Age Band and Extractor\n'
    '(bars = mean, error bars = 95% CI over 5 folds; CNN seeds averaged within each fold)',
    fontsize=13, fontweight='bold'
)
plt.tight_layout()
plt.savefig(root_dir / 'brainage_agg/outputs/figures/age_band_comparison.png',
            dpi=150, bbox_inches='tight')
plt.show()

## 6  Negative control validation (Arm B)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

ax = axes[0]
for agg, c in [('mean','#78909C'),('concatenation','#B0BEC5'),
                ('annualized_rate','#FB8C00'),('difference','#1E88E5'),('lme_slope_change','#7B1FA2')]:
    r2 = df[(df['target_arm']=='B') & (df['aggregation']==agg) & (df['age_band']=='all')]['R2'].clip(-5,1)
    ax.hist(r2, bins=30, alpha=0.6, color=c, label=f'{agg} μ={r2.mean():.3f}')
ax.axvline(0, color='red', linestyle='--', lw=1.5)
ax.set(xlabel='R²', ylabel='Count', title='Arm B R² distribution (0 = null predictor)')
ax.legend(fontsize=8)

ax = axes[1]
for agg, m, c, z in [('mean','o','#78909C',2),('concatenation','s','#B0BEC5',2),
                      ('annualized_rate','v','#FB8C00',3),('difference','^','#1E88E5',3),('lme_slope_change','D','#7B1FA2',3)]:
    sub = df[(df['target_arm']=='B') & (df['aggregation']==agg) & (df['age_band']=='all')]
    ax.scatter(sub['chance_mae'], sub['MAE'], alpha=0.3, s=12, marker=m, color=c, zorder=z,
               label=AGG_LABELS.get(agg,agg))
lims = [0, 1.6]
ax.plot(lims, lims, 'r--', lw=1.5, alpha=0.6, label='MAE = null predictor')
ax.set(xlim=lims, ylim=lims, xlabel='Null MAE (yrs)', ylabel='Model MAE (yrs)',
       title='Arm B: MAE vs chance (below diagonal = beats chance)')
ax.legend(fontsize=8, markerscale=2)
plt.tight_layout()
plt.savefig(root_dir / 'brainage_agg/outputs/figures/neg_control_validation.png', dpi=150, bbox_inches='tight')
plt.show()

print('Fraction of runs beating chance (MAE < chance_mae), Arm B, all bands:')
for agg in AGG_ORDER_B:
    sub = df[(df['target_arm']=='B') & (df['aggregation']==agg) & (df['age_band']=='all')]
    frac = (sub['MAE'] < sub['chance_mae']).mean()
    print(f'  {agg:25s}: {frac:.1%}  (n={len(sub)})')

## 7  Paired statistical tests

In [ ]:
def paired_test(arm, agg1, agg2, band='all'):
    sub = df[(df['target_arm']==arm) & (df['age_band']==band)]
    a1  = sub[sub['aggregation']==agg1][['extractor','fold','MAE']].rename(columns={'MAE':'m1'})
    a2  = sub[sub['aggregation']==agg2][['extractor','fold','MAE']].rename(columns={'MAE':'m2'})
    m   = a1.merge(a2, on=['extractor','fold'])
    if len(m) < 10: return float('nan'), float('nan'), 0
    stat, p = wilcoxon(m['m1'], m['m2'], alternative='less')
    return p, (m['m1']-m['m2']).mean(), len(m)

print(f'{"Arm":<4} {"Agg1":<22} {"Agg2":<22} {"p-value":>10} {"ΔMAE (a1-a2)":>14} {"n":>6}  sig')
print('-'*82)
for arm, a1, a2, note in [
    ('A','mean',             'lme_slope',          'state ≪ change (expected huge gap)'),
    ('A','concatenation',    'annualized_rate',     'concat vs rate'),
    ('A','mean',             'concatenation',       'mean vs concat'),
    ('B','annualized_rate',  'mean',                'annualized_rate vs state neg ctrl'),
    ('B','difference',       'mean',                'difference vs state neg ctrl'),
    ('B','lme_slope_change', 'mean',                'slope_change vs state neg ctrl'),
    ('B','annualized_rate',  'difference',          'annualized_rate vs difference'),
    ('B','annualized_rate',  'lme_slope_change',    'annualized_rate vs lme_slope_change'),
]:
    p, eff, n = paired_test(arm, a1, a2)
    sig = '***' if p<0.001 else '**' if p<0.01 else '*' if p<0.05 else 'ns'
    print(f'{arm:<4} {a1:<22} {a2:<22} {p:>10.2e} {eff:>14.4f} {n:>6}  {sig}  # {note}')

## 7b  CNN vs FS ROI — statistical test + variant bar plots

In [ ]:
# ── §7b  CNN vs FS ROI: Wilcoxon test + bar plots per variant ────────────────

# --- Statistical test: is CNN (avg over all seeds) better than FS ROI? --------
# Paired by CV fold (n=5); minimum achievable one-sided p ≈ 0.063.
print('Wilcoxon signed-rank: H₁  CNN MAE < FS MAE  (paired by CV fold, n=5)')
print(f"{'Arm':<4} {'Aggregation':<22} {'FS MAE':>8} {'CNN MAE':>8} {'ΔMAE':>8} {'p':>10}  sig  note")
print('─' * 88)

_stat_rows = []
for _arm in ['A', 'B']:
    for _agg in (AGG_ORDER_A if _arm == 'A' else AGG_ORDER_B):
        _sub = df[(df['target_arm']==_arm) & (df['age_band']=='all') & (df['aggregation']==_agg)]
        _fs  = _sub[_sub['extractor_type']=='fs_roi'].groupby('fold')['MAE'].mean()
        _cnn = _sub[_sub['extractor_type']=='cnn' ].groupby('fold')['MAE'].mean()
        _m   = pd.DataFrame({'fs': _fs, 'cnn': _cnn}).dropna()
        if len(_m) < 3:
            continue
        try:
            _p = wilcoxon(_m['cnn'].values, _m['fs'].values, alternative='less').pvalue
        except Exception:
            _p = float('nan')
        _delta = (_m['cnn'] - _m['fs']).mean()
        _sig   = '***' if _p < 0.001 else '**' if _p < 0.01 else '*' if _p < 0.05 else 'ns'
        _note  = '(CNN better)' if _delta < 0 else '(FS  better)'
        _stat_rows.append(dict(arm=_arm, agg=_agg, mae_fs=_m['fs'].mean(),
                               mae_cnn=_m['cnn'].mean(), delta=_delta, p=_p, sig=_sig))
        print(f"{_arm:<4} {_agg:<22} {_m['fs'].mean():>8.3f} {_m['cnn'].mean():>8.3f}"
              f" {_delta:>8.3f} {_p:>10.2e}  {_sig:<3}  {_note}")

print()
print('Note: with n=5 folds the minimum achievable one-sided p is ≈ 0.063;',
      'interpret "ns" cautiously when ΔMAE < 0 (CNN trending better).')

# --- Bar plots: all 24 CNN configs vs FS ROI baseline ------------------------
_CHANNEL_SHORT_7b = {'t1': 't1', 't1_sobel': 'sob',
                     't1_rank_sobel': 'rnk', 't1_median_sobel': 'med'}
_ARM_BEST_AGG = {'A': 'mean', 'B': 'annualized_rate'}

fig, axes = plt.subplots(1, 2, figsize=(20, 6))

for ax, arm in zip(axes, ['A', 'B']):
    agg    = _ARM_BEST_AGG[arm]
    fs_mae = df[(df['target_arm']==arm) & (df['age_band']=='all') &
                (df['aggregation']==agg) & (df['extractor_type']=='fs_roi')]['MAE'].mean()

    # CNN: average over 10 seeds per config (cnn_per_file already fold-averaged)
    _sub = cnn_per_file[(cnn_per_file['target_arm']==arm) &
                        (cnn_per_file['aggregation']==agg)].copy()
    _sub['config'] = (_sub['cnn_arch'].str[:3] + '/' + _sub['scaler'] + '/'
                      + _sub['channels'].map(_CHANNEL_SHORT_7b))
    cfg_mae = (_sub.groupby(['config', 'scaler'])['MAE']
               .mean().reset_index()
               .sort_values('MAE').reset_index(drop=True))

    best_cfg = cfg_mae.iloc[0]['config']
    bar_colors = [SCALER_COLORS.get(s, '#90CAF9')
                  for c, s in zip(cfg_mae['config'], cfg_mae['scaler'])]

    ax.bar(range(len(cfg_mae)), cfg_mae['MAE'], color=bar_colors,
           width=0.7, alpha=0.8, zorder=2, edgecolor='none')
    # Highlight best with black border
    ax.bar(0, cfg_mae.iloc[0]['MAE'], color=bar_colors[0],
           width=0.7, alpha=1.0, zorder=3, edgecolor='black', linewidth=1.8)
    ax.text(0, cfg_mae.iloc[0]['MAE'] + 0.03,
            f'★ {best_cfg}\n{cfg_mae.iloc[0]["MAE"]:.3f} yrs',
            ha='center', va='bottom', fontsize=7.5, fontweight='bold', color='#BF360C', zorder=5)

    ax.axhline(fs_mae, color='#1565C0', lw=2.5, ls='--', zorder=4)
    ax.text(len(cfg_mae) - 1, fs_mae + 0.03,
            f'FS ROI  {fs_mae:.3f} yrs',
            ha='right', va='bottom', fontsize=8.5, color='#1565C0', fontweight='bold')

    ax.set_xticks(range(len(cfg_mae)))
    ax.set_xticklabels(cfg_mae['config'], rotation=45, ha='right', fontsize=7.5)
    ax.set_ylabel('MAE (yrs, avg seeds × folds)')
    ax.set_title(f'Arm {arm} — {AGG_LABELS.get(agg, agg)} aggregation',
                 fontweight='bold', fontsize=12)
    ax.set_ylim(bottom=max(0, min(cfg_mae['MAE'].min() - 0.4, fs_mae - 0.4)))
    ax.grid(axis='y', alpha=0.35, zorder=0)

    # Scaler legend + FS baseline line
    _patches = [mpatches.Patch(color=SCALER_COLORS[s], label=s, alpha=0.85)
                for s in ['minmax', 'zscore', 'robust']]
    _best_patch = mpatches.Patch(color="#FFFFFF",  label='best config', linewidth=1.5,
                                 edgecolor='black')
    _fs_line = plt.Line2D([0], [0], color='#1565C0', lw=2.5, ls='--', label='FS ROI baseline')
    ax.legend(handles=_patches + [_best_patch, _fs_line],
              fontsize=8, loc='upper right', ncol=2)

plt.suptitle(
    'All 24 CNN configs vs FS ROI baseline — best config ★ highlighted\n'
    '(arch/scaler/channel, MAE averaged over 10 seeds × 5 folds)',
    fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(root_dir / 'brainage_agg/outputs/figures/cnn_variants_vs_fs.png',
            dpi=150, bbox_inches='tight')
plt.show()


---
# CNN Variants — Detailed Analysis

120 CNN files: **2 architectures** (cov_pool, double_conv) × **3 scalers** (minmax, zscore, robust) × **4 channel sets** (t1, t1_sobel, t1_rank_sobel, t1_median_sobel) × **10 seeds**.  
MAE values below are **averaged over 5 CV folds** for each file, then shown per configuration.

Key finding: **scaler is the dominant factor** — `robust` outperforms `minmax` by up to 2 MAE years.  
Worst outlier on Arm A: `cov_pool / minmax / t1 / seed=2` → MAE=1.44 yrs (scaler effect dominates).

> **Arm B target change:** Arm B now predicts **brain change rate** (PC1 of annualized FS ROI feature change), replacing the degenerate Δt ∈ {1, 2, 3} scan interval.  
> `annualized_rate` is now the primary expected-winner aggregation for Arm B alongside `difference` and `lme_slope_change`.  
> Results below will update once `sbatch brainage_agg/slurm/submit_run.sh` completes.


## 8  All 24 CNN configs — full overview heatmaps

In [ ]:
# Average over seeds (10 seeds → 1 value per arch × scaler × channels × agg)
cfg_mean = cnn_per_file.groupby(['cnn_arch','scaler','channels','aggregation','target_arm'])['MAE'].mean().reset_index()
cfg_mean['config'] = cfg_mean['cnn_arch'].str[:3] + '/' + cfg_mean['scaler'] + '/' + cfg_mean['channels']

CHANNEL_SHORT = {'t1':'t1','t1_sobel':'sob','t1_rank_sobel':'rnk','t1_median_sobel':'med'}
cfg_mean['config_short'] = (cfg_mean['cnn_arch'].str[:3] + '/' +
                             cfg_mean['scaler'] + '/' +
                             cfg_mean['channels'].map(CHANNEL_SHORT))

fig, axes = plt.subplots(1, 4, figsize=(22, 8))

for ax, arm, agg, title in [
    (axes[0], 'A', 'mean',          'Arm A — mean agg'),
    (axes[1], 'A', 'concatenation', 'Arm A — concat agg'),
    (axes[2], 'B', 'annualized_rate','Arm B — annualized rate agg'),
    (axes[3], 'B', 'lme_slope_change','Arm B — LME slope change'),
]:
    sub = cfg_mean[(cfg_mean['target_arm']==arm) & (cfg_mean['aggregation']==agg)].copy()
    pivot = sub.pivot_table(index='config_short', columns=None, values='MAE', aggfunc='first')

    # Build arch × scaler × channel 2D pivot
    sub['row'] = sub['cnn_arch'].str[:3] + '/' + sub['scaler']
    sub['col'] = sub['channels'].map(CHANNEL_SHORT)
    p2d = sub.pivot_table(index='row', columns='col', values='MAE')

    # Row order: cov/minmax, cov/robust, cov/zscore, dbl/minmax, dbl/robust, dbl/zscore
    row_order = ['cov/minmax','cov/robust','cov/zscore','dou/minmax','dou/robust','dou/zscore']
    col_order = ['t1','sob','rnk','med']
    p2d = p2d.reindex(index=[r for r in row_order if r in p2d.index],
                      columns=[c for c in col_order if c in p2d.columns])

    vmin, vmax = p2d.values.min()*0.97, p2d.values.max()*1.03
    sns.heatmap(p2d, annot=True, fmt='.2f', cmap='RdYlGn_r', ax=ax,
                vmin=vmin, vmax=vmax, linewidths=0.5, linecolor='white',
                cbar_kws={'label':'MAE (yrs avg/seed)','shrink':0.7},
                annot_kws={'size':9})
    ax.set_title(title, fontweight='bold', fontsize=11)
    ax.set_xlabel('Channels  (t1/sob=sobel/rnk=rank_sobel/med=median_sobel)')
    ax.set_ylabel('Arch/Scaler')
    ax.tick_params(axis='both', rotation=0)

plt.suptitle('CNN: All 24 configs (arch × scaler × channel) — MAE averaged over 10 seeds',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(root_dir / 'brainage_agg/outputs/figures/cnn_all_configs_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

## 9  Scaler effect — the dominant factor

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(20, 9))

# Share y-axis within each arm, per row
# Row 0 (boxplots): cols 0-1 = Arm A, cols 2-3 = Arm B
axes[0][1].sharey(axes[0][0])
axes[0][3].sharey(axes[0][2])
# Row 1 (bar charts): same grouping
axes[1][1].sharey(axes[1][0])
axes[1][3].sharey(axes[1][2])

cells = [
    ('A','mean',           'Arm A — mean'),
    ('A','concatenation',  'Arm A — concat'),
    ('B','annualized_rate','Arm B — ann. rate'),
    ('B','difference',      'Arm B — difference'),
]

for col, (arm, agg, title) in enumerate(cells):
    sub = cnn_per_file[(cnn_per_file['target_arm']==arm) & (cnn_per_file['aggregation']==agg)]

    # Top row: boxplots per scaler
    ax = axes[0][col]
    data_by_scaler = [sub[sub['scaler']==s]['MAE'].values for s in ['minmax','zscore','robust']]
    bp = ax.boxplot(data_by_scaler, labels=['minmax','zscore','robust'],
                    patch_artist=True, notch=False,
                    medianprops={'color':'black','lw':2})
    for patch, c in zip(bp['boxes'], [SCALER_COLORS['minmax'],
                                       SCALER_COLORS['zscore'],
                                       SCALER_COLORS['robust']]):
        patch.set_facecolor(c); patch.set_alpha(0.7)

    if arm == 'B':
        ax.axhline(df[(df['target_arm']=='B') & (df['age_band']=='all')]['chance_mae'].mean(),
                   color='red', linestyle='--', lw=1.5, alpha=0.7, label='Null: predict mean Δt')
        ax.legend(fontsize=8)

    fs_mae = df[(df['target_arm']==arm) & (df['aggregation']==agg) &
                (df['extractor_type']=='fs_roi') & (df['age_band']=='all')]['MAE'].mean()
    ax.axhline(fs_mae, color='#2196F3', linestyle=':', lw=2, alpha=0.8)
    ax.text(3.4, fs_mae, f'FSROI\n{fs_mae:.2f}', va='center', fontsize=7, color='#2196F3')

    ax.set_title(title, fontweight='bold', fontsize=10)
    # Only show y-label and tick labels on the left panel of each arm group
    ax.set_xlabel('Scaler')
    if col in (0, 2):
        ax.set_ylabel('MAE (yrs)')
    else:
        plt.setp(ax.get_yticklabels(), visible=False)

    # Bottom row: scaler × arch barplot
    ax2 = axes[1][col]
    arch_scaler = sub.groupby(['cnn_arch','scaler'])['MAE'].mean().reset_index()
    x, w = np.arange(3), 0.35
    for i, (arch, ac) in enumerate([('cov_pool','#5C6BC0'),('double_conv','#EF6C00')]):
        vals = [arch_scaler[(arch_scaler['cnn_arch']==arch) &
                             (arch_scaler['scaler']==s)]['MAE'].values for s in ['minmax','zscore','robust']]
        heights = [v[0] if len(v) else np.nan for v in vals]
        ax2.bar(x+i*w, heights, w, color=ac, alpha=0.8,
                label='cov_pool' if arch=='cov_pool' else 'double_conv')
        for xi, h in zip(x+i*w, heights):
            if not np.isnan(h):
                ax2.text(xi, h+h*0.005, f'{h:.2f}', ha='center', va='bottom', fontsize=7)

    ax2.set_xticks(x+w/2); ax2.set_xticklabels(['minmax','zscore','robust'])
    ax2.set_xlabel('Scaler')
    if col in (0, 2):
        ax2.set_ylabel('MAE (yrs)')
    else:
        plt.setp(ax2.get_yticklabels(), visible=False)

    if arm == 'B':
        ax2.axhline(df[(df['target_arm']=='B') & (df['age_band']=='all')]['chance_mae'].mean(),
                    color='red', linestyle='--', lw=1, alpha=0.7)
    if col == 3:
        ax2.legend(fontsize=8)

axes[0][0].set_ylabel('MAE (yrs) — all seeds')
axes[1][0].set_ylabel('MAE (yrs) — arch × scaler mean')
plt.suptitle('CNN: Scaler is the dominant factor  (blue dotted = FS ROI reference, red dashed = Arm B chance)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(root_dir / 'brainage_agg/outputs/figures/cnn_scaler_effect.png', dpi=150, bbox_inches='tight')
plt.show()

print('Scaler mean MAE (averaged over arch × channels × seeds):')
for arm, agg in [('A','mean'),('A','concatenation'),('B','difference'),('B','lme_slope_change')]:
    sub = cnn_per_file[(cnn_per_file['target_arm']==arm)&(cnn_per_file['aggregation']==agg)]
    row = sub.groupby('scaler')['MAE'].mean().round(3).to_dict()
    print(f'  Arm {arm} {agg:22s}: {row}')

## 10  Seed variability within each config (strip plots)

In [ ]:
# For Arm A mean and Arm B difference — show all 10 seeds per arch/scaler/channel config
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

for ax, arm, agg, title in [
    (axes[0], 'A', 'mean',       'Arm A — mean agg'),
    (axes[1], 'B', 'annualized_rate', 'Arm B — annualized rate agg'),
]:
    sub = cnn_per_file[(cnn_per_file['target_arm']==arm) & (cnn_per_file['aggregation']==agg)].copy()
    sub['config'] = sub['cnn_arch'].str[:3] + '/' + sub['scaler'] + '/' + sub['channels'].map(CHANNEL_SHORT)

    # Order configs by median MAE
    order = sub.groupby('config')['MAE'].median().sort_values().index.tolist()

    # Colour by scaler
    scaler_of_config = sub.drop_duplicates('config').set_index('config')['scaler']
    colors = [SCALER_COLORS[scaler_of_config[c]] for c in order]

    positions = np.arange(len(order))

    for pos, config, color in zip(positions, order, colors):
        vals = sub[sub['config']==config]['MAE'].values
        # Strip plot (jitter)
        jitter = np.random.default_rng(42).uniform(-0.15, 0.15, len(vals))
        ax.scatter(np.full(len(vals), pos) + jitter, vals,
                   color=color, alpha=0.8, s=25, zorder=3)
        # Median bar
        ax.hlines(np.median(vals), pos-0.3, pos+0.3, color=color, lw=2.5, zorder=4)

    # Chance / FS ROI reference lines
    if arm == 'B':
        ch = df[(df['target_arm']=='B')&(df['age_band']=='all')]['chance_mae'].mean()
        ax.axhline(ch, color='red', linestyle='--', lw=1.5, alpha=0.7, label=f'Null: predict mean Δt = {ch:.3f}')
    fs = df[(df['target_arm']==arm)&(df['aggregation']==agg)&
            (df['extractor_type']=='fs_roi')&(df['age_band']=='all')]['MAE'].mean()
    ax.axhline(fs, color='#2196F3', linestyle=':', lw=2, alpha=0.9, label=f'FS ROI {fs:.3f}')

    ax.set_xticks(positions)
    ax.set_xticklabels(order, rotation=55, ha='right', fontsize=7)
    ax.set(ylabel='MAE per seed (avg over folds)', title=title)
    ax.legend(fontsize=9)

    # Colour legend for scalers
    handles = [mpatches.Patch(color=SCALER_COLORS[s], label=s) for s in ['minmax','zscore','robust']]
    ax.legend(handles=handles + ax.get_legend_handles_labels()[0][-2:],
              fontsize=8, loc='upper left', ncol=2)

plt.suptitle('CNN: Seed variability per config (dots = seeds, bar = median; ordered by median MAE)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(root_dir / 'brainage_agg/outputs/figures/cnn_seed_variability_strips.png', dpi=150, bbox_inches='tight')
plt.show()

# Quantify: which configs have the highest seed variability?
for arm, agg in [('A','mean'),('B','difference')]:
    sub = cnn_per_file[(cnn_per_file['target_arm']==arm)&(cnn_per_file['aggregation']==agg)].copy()
    sub['config'] = sub['cnn_arch'].str[:3]+'/'+sub['scaler']+'/'+sub['channels'].map(CHANNEL_SHORT)
    spread = sub.groupby('config')['MAE'].agg(['mean','std','min','max']).sort_values('std', ascending=False)
    spread['range'] = spread['max'] - spread['min']
    print(f'\nTop-5 most variable configs — Arm {arm} {agg}:')
    print(spread.head(5).round(3).to_string())

## 11  Arm B annualized rate — instability analysis


In [ ]:
sub_diff = cnn_per_file[
    (cnn_per_file['target_arm']=='B') & (cnn_per_file['aggregation']=='annualized_rate')
].copy()
chance = df[(df['target_arm']=='B')&(df['age_band']=='all')]['chance_mae'].mean()

fig, axes = plt.subplots(1, 3, figsize=(17, 5))

# Left: per-seed MAE, coloured by scaler; marker=arch
ax = axes[0]
for arch, marker in [('cov_pool','o'),('double_conv','s')]:
    for scaler, c in SCALER_COLORS.items():
        pts = sub_diff[(sub_diff['cnn_arch']==arch)&(sub_diff['scaler']==scaler)]
        ax.scatter(pts['cnn_seed']+0.15*(list(SCALER_COLORS).index(scaler)-1),
                   pts['MAE'], color=c, marker=marker, alpha=0.75, s=40)
ax.axhline(chance, color='red', linestyle='--', lw=1.5, label=f'Null: predict median brain change rate = {chance:.3f}')
ax.set(xlabel='CNN seed', ylabel='MAE (yrs)', title='Arm B ann. rate: all seeds\n(marker=arch, color=scaler)',
       xticks=range(10))
handles  = [mpatches.Patch(color=c, label=s) for s, c in SCALER_COLORS.items()]
handles += [plt.Line2D([0],[0],marker='o',color='grey',ls='',label='cov_pool'),
            plt.Line2D([0],[0],marker='s',color='grey',ls='',label='double_conv')]
ax.legend(handles=handles, fontsize=8, ncol=2)

# Middle: distribution of MAE coloured by arch × scaler
ax = axes[1]
for arch, ls in [('cov_pool','-'),('double_conv','--')]:
    for scaler, c in SCALER_COLORS.items():
        vals = sub_diff[(sub_diff['cnn_arch']==arch)&(sub_diff['scaler']==scaler)]['MAE']
        ax.hist(vals, bins=12, alpha=0.5, color=c, linestyle=ls,
                label=f'{arch[:3]}/{scaler}', density=True)
ax.axvline(chance, color='red', linestyle='--', lw=1.5)
ax.set(xlabel='MAE (yrs)', ylabel='Density',
       title='Distribution per arch/scaler\n(all channels × seeds)')
ax.legend(fontsize=7, ncol=2)

# Right: channel effect — violin per channel × scaler
ax = axes[2]
parts = ax.violinplot(
    [sub_diff[sub_diff['channels']==ch]['MAE'].values
     for ch in ['t1','t1_sobel','t1_rank_sobel','t1_median_sobel']],
    positions=range(4), showmedians=True
)
for pc, c in zip(parts['bodies'], CHANNEL_COLORS.values()):
    pc.set_facecolor(c); pc.set_alpha(0.6)
ax.axhline(chance, color='red', linestyle='--', lw=1.5)
ax.set_xticks(range(4))
ax.set_xticklabels(['t1','t1_sobel','t1_rank\n_sobel','t1_med\n_sobel'], fontsize=8)
ax.set(ylabel='MAE (yrs)', title='Channel effect\n(all arch × scaler × seeds)')

plt.suptitle('Arm B — Difference aggregation: instability analysis across CNN variants',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(root_dir / 'brainage_agg/outputs/figures/cnn_armb_instability.png', dpi=150, bbox_inches='tight')
plt.show()

# Highlight outliers
outliers = sub_diff[sub_diff['MAE'] > chance * 1.2].sort_values('MAE', ascending=False)
print(f'\nOutlier configs (MAE > 1.2× chance = {chance*1.2:.3f} yrs):')
print(outliers[['cnn_arch','scaler','channels','cnn_seed','MAE']].head(10).to_string(index=False))

## 12  Arm A concat — best CNN configs vs FS ROI

In [ ]:
# Concat can beat FS ROI for Arm A — identify when
sub_cat = cnn_per_file[
    (cnn_per_file['target_arm']=='A') & (cnn_per_file['aggregation']=='concatenation')
].copy()

fs_concat_mae = df[
    (df['target_arm']=='A')&(df['aggregation']=='concatenation')&
    (df['extractor_type']=='fs_roi')&(df['age_band']=='all')
]['MAE'].mean()

sub_cat['beats_fs'] = sub_cat['MAE'] < fs_concat_mae

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Left: all seeds per config, line = FS ROI
ax = axes[0]
sub_cat2 = sub_cat.copy()
sub_cat2['config'] = sub_cat2['cnn_arch'].str[:3]+'/'+sub_cat2['scaler']+'/'+sub_cat2['channels'].map(CHANNEL_SHORT)
order = sub_cat2.groupby('config')['MAE'].median().sort_values().index.tolist()

for pos, config in enumerate(order):
    vals = sub_cat2[sub_cat2['config']==config]['MAE'].values
    scaler = config.split('/')[1]
    color  = SCALER_COLORS[scaler]
    jitter = np.random.default_rng(0).uniform(-0.15, 0.15, len(vals))
    ax.scatter(np.full(len(vals), pos)+jitter, vals, color=color, alpha=0.75, s=25)
    ax.hlines(np.median(vals), pos-0.3, pos+0.3, color=color, lw=2)

ax.axhline(fs_concat_mae, color='#2196F3', linestyle=':', lw=2.5,
           label=f'FS ROI concat {fs_concat_mae:.3f}')
ax.fill_between([-0.5, len(order)-0.5], [0,0], [fs_concat_mae, fs_concat_mae],
                color='#2196F3', alpha=0.07)
ax.set_xticks(range(len(order)))
ax.set_xticklabels(order, rotation=55, ha='right', fontsize=7)
ax.set(ylabel='MAE (yrs)', title=f'Arm A concat: all seeds per config\n(shaded = beats FS ROI)')
handles = [mpatches.Patch(color=c, label=s) for s, c in SCALER_COLORS.items()]
handles.append(plt.Line2D([0],[0],color='#2196F3',ls=':',lw=2,label='FS ROI'))
ax.legend(handles=handles, fontsize=8)

# Right: fraction of seeds beating FS ROI per config
ax = axes[1]
sub_cat2['beats_fs'] = sub_cat2['MAE'] < fs_concat_mae
frac = sub_cat2.groupby('config')['beats_fs'].mean().reindex(order)
colors = [SCALER_COLORS[c.split('/')[1]] for c in order]
bars = ax.barh(range(len(order)), frac.values, color=colors, alpha=0.8)
ax.axvline(0.5, color='grey', linestyle='--', lw=1)
ax.set_yticks(range(len(order)))
ax.set_yticklabels(order, fontsize=7)
ax.set(xlabel='Fraction of seeds beating FS ROI concat',
       title='How often does each config\nbeat FS ROI concat?')
ax.set_xlim(0, 1)
for i, (bar, v) in enumerate(zip(bars, frac.values)):
    ax.text(v+0.01, i, f'{v:.0%}', va='center', fontsize=7)

plt.suptitle('Arm A Concatenation: CNN vs FS ROI (blue zone = below FS ROI)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(root_dir / 'brainage_agg/outputs/figures/cnn_concat_vs_fsroi.png', dpi=150, bbox_inches='tight')
plt.show()

frac_beats = (sub_cat['MAE'] < fs_concat_mae).mean()
print(f'FS ROI concat MAE = {fs_concat_mae:.3f}')
print(f'CNN concat seeds beating FS ROI: {frac_beats:.1%} ({(sub_cat["MAE"] < fs_concat_mae).sum()}/{len(sub_cat)})')
print('\nBest 5 CNN concat configs:')
print(sub_cat.nsmallest(5, 'MAE')[['cnn_arch','scaler','channels','cnn_seed','MAE']].to_string(index=False))

## 13  Cross-arm consistency — does the same config win in both arms?

In [ ]:
# Compare MAE rank for each CNN file in Arm A (mean) vs Arm B (difference)
arm_a_mean = cnn_per_file[(cnn_per_file['target_arm']=='A') & (cnn_per_file['aggregation']=='mean')][['cnn_arch','scaler','channels','cnn_seed','MAE']].rename(columns={'MAE':'mae_a'})
arm_b_diff = cnn_per_file[(cnn_per_file['target_arm']=='B') & (cnn_per_file['aggregation']=='difference')][['cnn_arch','scaler','channels','cnn_seed','MAE']].rename(columns={'MAE':'mae_b'})
cross = arm_a_mean.merge(arm_b_diff, on=['cnn_arch','scaler','channels','cnn_seed'])
cross['config'] = cross['cnn_arch'].str[:3]+'/'+cross['scaler']+'/'+cross['channels'].map(CHANNEL_SHORT)

spearman_r, spearman_p = stats.spearmanr(cross['mae_a'], cross['mae_b'])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
for scaler, c in SCALER_COLORS.items():
    sub = cross[cross['scaler']==scaler]
    ax.scatter(sub['mae_a'], sub['mae_b'], alpha=0.7, s=35, color=c,
               label=f'{scaler} (n={len(sub)})')
    # Convex hull / ellipse would be nice, skip for simplicity

# Regression line
m, b, *_ = stats.linregress(cross['mae_a'], cross['mae_b'])
xr = np.linspace(cross['mae_a'].min(), cross['mae_a'].max(), 100)
ax.plot(xr, m*xr+b, 'k--', lw=1.5, alpha=0.6)
ax.set(xlabel='Arm A mean agg MAE (yrs)', ylabel='Arm B difference agg MAE (yrs)',
       title=f'Cross-arm consistency\nSpearman r={spearman_r:.3f}, p={spearman_p:.2e}')
ax.legend(fontsize=9)

# Right: per-config mean in both arms — bar chart
ax = axes[1]
cfg_cross = cross.groupby('config')[['mae_a','mae_b']].mean().sort_values('mae_a')
x = np.arange(len(cfg_cross))
ax.bar(x-0.2, cfg_cross['mae_a'].values, 0.4, label='Arm A mean', alpha=0.75,
       color=[SCALER_COLORS[c.split('/')[1]] for c in cfg_cross.index])
ax2 = ax.twinx()
ax2.bar(x+0.2, cfg_cross['mae_b'].values, 0.4, label='Arm B diff', alpha=0.45,
        color=[SCALER_COLORS[c.split('/')[1]] for c in cfg_cross.index])
ax.set_xticks(x)
ax.set_xticklabels(cfg_cross.index.tolist(), rotation=55, ha='right', fontsize=6.5)
ax.set_ylabel('MAE Arm A (yrs)', color='black')
ax2.set_ylabel('MAE Arm B (yrs)', color='grey')
ax.set_title('Config ranking: Arm A vs Arm B\n(sorted by Arm A MAE, color=scaler)')
lines1, labels1 = ax.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax.legend(lines1+lines2, labels1+labels2, fontsize=8, loc='upper left')

plt.suptitle('Does the same CNN config win in both arms?', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(root_dir / 'brainage_agg/outputs/figures/cnn_cross_arm_consistency.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Spearman r = {spearman_r:.3f}  (p={spearman_p:.2e})')
print('Interpretation: ', end='')
if abs(spearman_r) > 0.5: print('strong agreement — good Arm A configs tend to be good for Arm B too')
elif abs(spearman_r) > 0.2: print('moderate agreement — partial overlap in what makes a good config')
else: print('weak/no agreement — what works for Arm A does NOT predict Arm B performance')

## 14  Annualized rate — extreme outliers investigation

In [ ]:
sub_rate = cnn_per_file[
    (cnn_per_file['target_arm']=='A') & (cnn_per_file['aggregation']=='annualized_rate')
].copy()

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Distribution — log scale to show the extreme tail
ax = axes[0]
ax.hist(sub_rate['MAE'].clip(0, 30), bins=40, color='#AB47BC', alpha=0.8, edgecolor='white')
ax.axvline(df[(df['target_arm']=='A')&(df['aggregation']=='mean')&
              (df['extractor_type']=='cnn')&(df['age_band']=='all')]['MAE'].mean(),
           color='#43A047', linestyle='--', lw=2, label='CNN mean MAE')
ax.set(xlabel='MAE (yrs, clipped at 30)', ylabel='Count',
       title=f'Arm A annualized_rate MAE distribution\n(full range: {sub_rate["MAE"].min():.1f}–{sub_rate["MAE"].max():.1f} yrs)')
ax.legend(fontsize=8)

# Extremes by scaler
ax = axes[1]
for scaler, c in SCALER_COLORS.items():
    vals = sub_rate[sub_rate['scaler']==scaler]['MAE']
    ax.scatter(np.full(len(vals), list(SCALER_COLORS).index(scaler)), vals.clip(0,30),
               color=c, alpha=0.6, s=30, label=scaler)
    ax.hlines(vals.clip(0,30).median(), list(SCALER_COLORS).index(scaler)-0.3,
              list(SCALER_COLORS).index(scaler)+0.3, color=c, lw=2.5)
ax.set(xticks=range(3), xticklabels=list(SCALER_COLORS.keys()),
       ylabel='MAE (yrs, clipped at 30)', title='Rate agg instability by scaler')
ax.legend(fontsize=8)

# Correlation: rate MAE vs mean MAE (same config)
ax = axes[2]
arm_a_rate = sub_rate[['cnn_arch','scaler','channels','cnn_seed','MAE']].rename(columns={'MAE':'mae_rate'})
arm_a_mean_sub = cnn_per_file[(cnn_per_file['target_arm']=='A') & (cnn_per_file['aggregation']=='mean')][['cnn_arch','scaler','channels','cnn_seed','MAE']].rename(columns={'MAE':'mae_mean'})
rate_vs_mean = arm_a_rate.merge(arm_a_mean_sub, on=['cnn_arch','scaler','channels','cnn_seed'])
for scaler, c in SCALER_COLORS.items():
    sub = rate_vs_mean[rate_vs_mean['scaler']==scaler]
    ax.scatter(sub['mae_mean'], sub['mae_rate'].clip(0,30), alpha=0.7, s=30, color=c, label=scaler)
ax.set(xlabel='MAE — mean agg (yrs)', ylabel='MAE — annualized rate (yrs, clip@30)',
       title='Rate vs mean aggregation\n(same CNN config)')
ax.legend(fontsize=8)

plt.suptitle('Arm A Annualized Rate: instability deep-dive', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(root_dir / 'brainage_agg/outputs/figures/cnn_rate_instability.png', dpi=150, bbox_inches='tight')
plt.show()

print('Annualized rate: fraction of configs with MAE > 20 yrs (extreme failure):')
for scaler in ['minmax','zscore','robust']:
    vals = sub_rate[sub_rate['scaler']==scaler]['MAE']
    pct = (vals > 20).mean()
    print(f'  {scaler:8s}: {pct:.1%}  ({(vals>20).sum()}/{len(vals)}) — max={vals.max():.1f}')

## 15  Full factorial metrics heatmap

In [ ]:
metrics = [
    ('MAE',  'MAE (yrs)',    True),
    ('R2',   'R²',           False),
    ('r',    'Pearson r',    False),
    ('RMSE', 'RMSE (yrs)',   True),
]

all_band = df[df['age_band']=="all"].copy()
all_band['ext_label'] = all_band['extractor_type'].map({'fs_roi':'FS ROI','cnn':'CNN'})

for arm, agg_order, arm_label in [
    ('A', AGG_ORDER_A, 'Arm A — Absolute age prediction'),
    ('B', AGG_ORDER_B, 'Arm B — Brain change rate prediction'),
]:
    fig, axes = plt.subplots(1, 4, figsize=(22, 4))

    sub = all_band[all_band['target_arm']==arm]
    cell_vals = sub.groupby(['aggregation','ext_label'])[
        [m for m,_,_ in metrics]
    ].mean().reset_index()

    for ax, (metric, ylabel, lower_better) in zip(axes, metrics):
        pivot = cell_vals.pivot(index='aggregation', columns='ext_label', values=metric)
        pivot = pivot.reindex([a for a in agg_order if a in pivot.index])
        pivot.index = [AGG_LABELS.get(a, a) for a in pivot.index]

        # Per-arm, per-metric color scale
        vmin, vmax = pivot.values.min(), pivot.values.max()
        cmap = 'RdYlGn_r' if lower_better else 'RdYlGn'

        sns.heatmap(
            pivot, annot=True, fmt='.3f', cmap=cmap, ax=ax,
            vmin=vmin, vmax=vmax,
            linewidths=0.5, linecolor='white',
            cbar_kws={'label': ylabel, 'shrink': 0.8},
            annot_kws={'size': 10, 'weight': 'bold'},
        )
        ax.set_title(ylabel, fontweight='bold')
        ax.set(xlabel='', ylabel='')
        ax.tick_params(axis='both', rotation=0)

    plt.suptitle(arm_label + '  (CNN = mean over all variants)',
                 fontsize=13, fontweight='bold')
    plt.tight_layout()
    fname = f'brainage_agg/outputs/figures/full_factorial_metrics_arm{arm}.png'
    plt.savefig(root_dir / fname, dpi=150, bbox_inches='tight')
    plt.show()

---
## 15b  Child vs Adult classification

Cross-sectional and longitudinal aggregations used as inputs to Logistic Regression and Random Forest.  
Target: `band` (Child=0, Adult=1). Metric: balanced accuracy (primary) and AUC-ROC.  
Same 5-fold GroupKFold CV as the regression arms.

**Aggregations tested:**
- `cross_sectional` — first timepoint only (no longitudinal information, pure anatomy)
- `mean` — element-wise mean across timepoints (state)
- `concatenation` — [f_first, f_mid, f_last], requires ≥3 tp
- `annualized_rate` — (f_last − f_first) / Δt (change per year; no circularity here since Δt is not the target)
- `difference` — f_last − f_first (raw change)
- `lme_slope_change` — per-subject OLS slope (regularized change descriptor)

In [ ]:
# Load from merged SLURM results if available, else run inline (FS ROI only)
import sys; sys.path.insert(0, str(root_dir))
from brainage_agg.data.manifest import build_manifest
from brainage_agg.features.loader import load_features, align_to_manifest
from brainage_agg.agg import aggregations as agg_mod
from brainage_agg.agg.lme import VectorizedOLSSlopes
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupKFold
from sklearn.metrics import balanced_accuracy_score, roc_auc_score, f1_score

_merged = root_dir / 'brainage_agg/outputs/classif_results.csv'

if _merged.exists():
    classif_df = pd.read_csv(_merged)
    print(f'Loaded merged results: {len(classif_df)} rows from {_merged}')
    print(f'Extractors: {classif_df["extractor_type"].value_counts().to_dict()}')
else:
    print('classif_results.csv not found — running inline on FS ROI only.')
    print('To run all 241 extractors in parallel:')
    print('  sbatch brainage_agg/experiment/submit_classification.sh')
    print('  python brainage_agg/experiment/run_classification.py --merge')
    print()

    # ── helpers (same logic as run_classification.py) ──────────────────────
    CLASSIF_AGGS = ['cross_sectional','mean','concatenation',
                    'annualized_rate','difference','lme_slope_change']

    def _flat_lme_inputs(sids, subject_data, scaler):
        feats = np.vstack([subject_data[s]['features'] for s in sids])
        ages  = np.concatenate([subject_data[s]['ages']     for s in sids])
        ids   = np.concatenate([[s]*len(subject_data[s]['ages']) for s in sids])
        return scaler.transform(feats), ages, ids

    def _build_subject_matrix(mdf, subject_data, aggregation, lme_subs=None, lme_slopes=None):
        rows_X, rows_y, groups = [], [], []
        for _, row in mdf.iterrows():
            sid  = row['subject_id']
            if sid not in subject_data: continue
            band = row['band']
            if band not in ('Child','Adult'): continue
            data  = subject_data[sid]
            feats = data['features']; ages = data['ages']
            y_val = 0 if band == 'Child' else 1
            if   aggregation == 'cross_sectional': x = feats[0]
            elif aggregation == 'mean':             x = agg_mod.mean(feats)
            elif aggregation == 'concatenation':
                if row['t_mid_idx'] < 0: continue
                t_mid_local = list(row['row_indices']).index(row['t_mid_idx'])
                x = agg_mod.concatenation(feats, 0, t_mid_local, len(feats)-1)
            elif aggregation == 'annualized_rate':
                dt = float(ages[-1]-ages[0])
                if abs(dt)<1e-8: continue
                x = agg_mod.annualized_rate(feats, ages)
            elif aggregation == 'difference':       x = agg_mod.difference(feats)
            elif aggregation == 'lme_slope_change':
                if lme_subs is None: continue
                x = agg_mod.lme_slope(str(sid), lme_subs, lme_slopes)
            else: continue
            rows_X.append(x); rows_y.append(y_val); groups.append(sid)
        if not rows_X:
            return np.empty((0,0)), np.empty(0,int), np.empty(0,object)
        return np.array(rows_X,dtype=np.float32), np.array(rows_y,int), np.array(groups)

    def run_classif_cv(manifest_df, subject_data, aggregation, extractor_label, n_folds=5):
        spec   = agg_mod.AGGREGATION_SPECS.get(aggregation, {})
        cohort = 'cohort_concat' if spec.get('cohort')=='concat' else 'cohort_all'
        mdf    = manifest_df[manifest_df[cohort] & manifest_df['band'].isin(['Child','Adult'])].copy()
        if spec.get('needs_ages') or aggregation in ('difference','lme_slope_change'):
            mdf = mdf[~mdf['exclude_arm_b']]
        sids_all = [r.subject_id for r in mdf.itertuples()
                    if r.subject_id in subject_data and r.band in ('Child','Adult')]
        if len(sids_all) < n_folds*4: return []
        dummy_X = np.zeros((len(sids_all),1))
        rows = []
        for fold,(tr_idx,te_idx) in enumerate(GroupKFold(n_splits=n_folds).split(dummy_X,groups=sids_all)):
            tr_sids = [sids_all[i] for i in tr_idx]
            te_sids = [sids_all[i] for i in te_idx]
            scaler  = StandardScaler().fit(np.vstack([subject_data[s]['features'] for s in tr_sids]))
            sd_sc   = {s:{'features':scaler.transform(subject_data[s]['features']),
                          'ages':subject_data[s]['ages'],
                          'row_indices':subject_data[s]['row_indices']}
                       for s in set(tr_sids+te_sids)}
            lme_subs=lme_slopes=None
            if aggregation=='lme_slope_change':
                lf,la,li = _flat_lme_inputs(tr_sids,subject_data,scaler)
                lme_model= VectorizedOLSSlopes().fit(lf,la,li)
                all_s    = list(dict.fromkeys(tr_sids+te_sids))
                af,aa,ai = _flat_lme_inputs(all_s,subject_data,scaler)
                lme_subs,lme_slopes = lme_model.transform(af,aa,ai)
            X_tr,y_tr,_= _build_subject_matrix(mdf[mdf['subject_id'].isin(tr_sids)],sd_sc,aggregation,lme_subs,lme_slopes)
            X_te,y_te,_= _build_subject_matrix(mdf[mdf['subject_id'].isin(te_sids)],sd_sc,aggregation,lme_subs,lme_slopes)
            if len(X_tr)<4 or len(np.unique(y_tr))<2 or len(np.unique(y_te))<2: continue
            for clf_name,clf in [
                ('LogisticReg', LogisticRegression(C=1.0,max_iter=2000,class_weight='balanced')),
                ('RandomForest',RandomForestClassifier(n_estimators=200,max_features='sqrt',
                                                        class_weight='balanced',random_state=42,n_jobs=2)),
            ]:
                clf.fit(X_tr,y_tr); pred=clf.predict(X_te); prob=clf.predict_proba(X_te)[:,1]
                rows.append(dict(extractor_type='fs_roi',extractor=extractor_label,
                                 cnn_arch='',scaler='',channels='',cnn_seed='',
                                 aggregation=aggregation,classifier=clf_name,fold=fold,
                                 bacc=float(balanced_accuracy_score(y_te,pred)),
                                 auc=float(roc_auc_score(y_te,prob)),
                                 f1_macro=float(f1_score(y_te,pred,average='macro')),
                                 n_train=len(y_tr),n_test=len(y_te)))
        return rows

    fs_npz = root_dir / 'outputs/features/features__model-freesurfer_roi__scaler-none__channels-all_roi__seed-0.npz'
    manifest_cl = build_manifest(fs_npz, t_mid_seed=42)
    X_fs, meta_fs, _ = load_features(fs_npz)
    sd_fs = align_to_manifest(X_fs, meta_fs, manifest_cl)

    classif_rows = []
    for agg in CLASSIF_AGGS:
        r = run_classif_cv(manifest_cl, sd_fs, agg, extractor_label='fs_roi')
        classif_rows.extend(r)
        print(f'  {agg}: {len(r)} rows')
    classif_df = pd.DataFrame(classif_rows)

print(f'classif_df: {len(classif_df)} rows | extractors: {classif_df["extractor_type"].value_counts().to_dict()}')

In [ ]:
AGG_ORDER_CL = ['cross_sectional','mean','concatenation','annualized_rate','difference','lme_slope_change']
AGG_LABELS_CL = {
    'cross_sectional': 'Cross-\nsectional',
    'mean':            'Mean',
    'concatenation':   'Concat',
    'annualized_rate': 'Ann.\nRate',
    'difference':      'Difference',
    'lme_slope_change':'LME\nSlope',
}
CLF_COLORS = {'LogisticReg': '#1976D2', 'RandomForest': '#E64A19'}

# ── Figure 1: FS ROI — balanced accuracy and AUC by aggregation × classifier ──
fs_df = classif_df[classif_df['extractor_type']=='fs_roi'].copy()
cnn_cl = classif_df[classif_df['extractor_type']=='cnn'].copy()
summary_fs = (fs_df.groupby(['aggregation','classifier'])[['bacc','auc']]
              .agg(['mean','std']).reset_index())
summary_fs.columns = ['aggregation','classifier','bacc_mean','bacc_std','auc_mean','auc_std']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
x = np.arange(len(AGG_ORDER_CL))
w = 0.35

for ax, metric, metric_std, ylabel, title in [
    (axes[0], 'bacc_mean', 'bacc_std', 'Balanced Accuracy', 'Balanced Accuracy'),
    (axes[1], 'auc_mean',  'auc_std',  'AUC-ROC',           'AUC-ROC'),
]:
    for i, clf in enumerate(['LogisticReg', 'RandomForest']):
        sub = summary_fs[summary_fs['classifier']==clf].set_index('aggregation')
        vals = [sub.loc[a, metric] if a in sub.index else np.nan for a in AGG_ORDER_CL]
        errs = [sub.loc[a, metric_std] if a in sub.index else 0  for a in AGG_ORDER_CL]
        bars = ax.bar(x + i*w, vals, w, label=clf, color=CLF_COLORS[clf],
                      alpha=0.85, yerr=errs, capsize=4, error_kw={'lw':1.5})
        for bar, v in zip(bars, vals):
            if not np.isnan(v):
                ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.004,
                        f'{v:.2f}', ha='center', va='bottom', fontsize=7, fontweight='bold')

    ax.axhline(0.5, color='grey', linestyle='--', lw=1.2, alpha=0.7, label='Chance (0.5)')
    ax.set_xticks(x+w/2)
    ax.set_xticklabels([AGG_LABELS_CL.get(a,a) for a in AGG_ORDER_CL], fontsize=9)
    ax.set(ylabel=ylabel, title=f'FS ROI — {title}', ylim=(0.25, 1.05))
    ax.legend(fontsize=9, loc='lower left')
    ax.vlines(2.7, 0, 1.05, color='grey', linestyle=':', lw=0.8, alpha=0.7)


plt.suptitle('Child vs Adult Classification — FS ROI features  (green=state, blue=change)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(root_dir / 'brainage_agg/outputs/figures/classif_fsroi.png', dpi=150, bbox_inches='tight')
plt.show()



# ── Figure 3: Best CNN config per scaler ────────────────────────────────────
if len(cnn_cl) > 0:
    # Average over CV folds per extractor file × aggregation × classifier
    cnn_cl_per_file = (cnn_cl.groupby(['cnn_arch','scaler','channels','cnn_seed','aggregation','classifier'])
                    ['bacc'].mean().reset_index())
    aggs_shown  = ['cross_sectional','mean','concatenation','annualized_rate','difference','lme_slope_change']
    scalers     = ['minmax','zscore','robust']
    scaler_colors = {'minmax':'#EF5350','zscore':'#FFA726','robust':'#66BB6A'}
    x = np.arange(len(aggs_shown))
    w = 0.25

    # For each (scaler, aggregation, classifier): pick the seed/arch/channel with highest bacc
    best_per_scaler = (cnn_cl_per_file
                       .groupby(['scaler','aggregation','classifier'])['bacc']
                       .max().reset_index()
                       .rename(columns={'bacc':'best_bacc'}))
    # Also track which config achieved the best (for annotation)
    idx_best = (cnn_cl_per_file
                .groupby(['scaler','aggregation','classifier'])['bacc']
                .idxmax())
    best_configs = cnn_cl_per_file.loc[idx_best][
        ['scaler','aggregation','classifier','cnn_arch','channels','cnn_seed','bacc']
    ].rename(columns={'bacc':'best_bacc'}).reset_index(drop=True)

    # Pre-compute fold-level bacc for each best config (sign test vs FS ROI)
    _best_cnn_folds = {}  # (scaler, agg, clf) -> fold-level bacc array
    for _, _bc in best_configs.iterrows():
        _k = (_bc['scaler'], _bc['aggregation'], _bc['classifier'])
        _fc = cnn_cl[
            (cnn_cl['cnn_arch']    == _bc['cnn_arch'])   &
            (cnn_cl['scaler']      == _bc['scaler'])      &
            (cnn_cl['channels']    == _bc['channels'])    &
            (cnn_cl['cnn_seed']    == _bc['cnn_seed'])    &
            (cnn_cl['aggregation'] == _bc['aggregation']) &
            (cnn_cl['classifier']  == _bc['classifier'])
        ]
        _best_cnn_folds[_k] = _fc.sort_values('fold')['bacc'].values

    fig, axes = plt.subplots(1, 2, figsize=(16, 5), sharey=True)
    for ax, clf in zip(axes, ['LogisticReg', 'RandomForest']):
        sub_dist = cnn_cl_per_file[cnn_cl_per_file['classifier']==clf]
        sub_best = best_configs[best_configs['classifier']==clf]
        _fs_folds = {a: fs_df[(fs_df['aggregation']==a) & (fs_df['classifier']==clf)
                               ].sort_values('fold')['bacc'].values for a in aggs_shown}

        for i, scaler in enumerate(scalers):
            positions = x + (i - 1) * w

            # Grey boxes showing distribution (same as Fig 3, muted)
            data = [sub_dist[sub_dist['scaler']==scaler][
                        sub_dist['aggregation']==a]['bacc'].values for a in aggs_shown]
            bp = ax.boxplot(data, positions=positions, widths=w*0.8,
                            patch_artist=True, notch=False,
                            medianprops={'color':'grey','lw':1},
                            whiskerprops={'lw':0.8,'color':'grey'},
                            capprops={'lw':0.8,'color':'grey'},
                            flierprops={'marker':'.','markersize':2,'alpha':0.2,'color':'grey'})
            for patch in bp['boxes']:
                patch.set_facecolor('#EEEEEE'); patch.set_alpha(0.5)

            # Best value as a filled diamond
            s_best = sub_best[sub_best['scaler']==scaler]
            for xi, a in enumerate(aggs_shown):
                row = s_best[s_best['aggregation']==a]
                if len(row):
                    bv = row['best_bacc'].values[0]
                    ax.scatter(xi + (i-1)*w, bv, marker='D', s=60,
                               color=scaler_colors[scaler], zorder=5,
                               edgecolors='black', linewidths=0.5)
                    ax.text(xi + (i-1)*w, bv + 0.008, f'{bv:.3f}',
                            ha='center', va='bottom', fontsize=6,
                            color=scaler_colors[scaler], fontweight='bold')
                    # Sign test: H₁ CNN best > FS ROI (one-sided, paired by fold)
                    _cnn_f = _best_cnn_folds.get((scaler, a, clf), np.array([]))
                    _fs_f  = _fs_folds.get(a, np.array([]))
                    if (len(_cnn_f) > 0 and len(_fs_f) == len(_cnn_f)
                            and bv > _fs_f.mean()):
                        _n_better = int(np.sum(_cnn_f > _fs_f))
                        _p_sign   = scipy_stats.binom.sf(_n_better - 1, len(_cnn_f), 0.5)
                        if _p_sign < 0.05:
                            ax.text(xi + (i-1)*w, bv + 0.022, '*',
                                    ha='center', va='bottom', fontsize=14,
                                    color='black', fontweight='bold')

        # FS ROI reference
        fs_vals = [summary_fs[(summary_fs['classifier']==clf)&(summary_fs['aggregation']==a)
                              ]['bacc_mean'].values for a in aggs_shown]
        for xi, fv in enumerate(fs_vals):
            if len(fv):
                ax.hlines(fv[0], xi-0.45, xi+0.45, color='black', lw=2,
                          linestyle=':', label='FS ROI' if xi==0 else '')

        ax.axhline(0.5, color='grey', linestyle='--', lw=1, alpha=0.6)
        ax.set_xticks(x)
        ax.set_xticklabels([AGG_LABELS_CL.get(a,a) for a in aggs_shown])
        ax.set(ylabel='Balanced Accuracy' if clf=='LogisticReg' else '',
               title=clf, ylim=(0.4, 1.05))

        handles = ([mpatches.Patch(facecolor=scaler_colors[s], alpha=0.9, label=s)
                    for s in scalers] +
                   [plt.Line2D([0],[0], marker='D', color='w', markerfacecolor='grey',
                               markeredgecolor='black', markersize=7, label='Best config'),
                    plt.Line2D([0],[0], color='black', lw=2, linestyle=':', label='FS ROI')])
        ax.legend(handles=handles, fontsize=9, loc='lower left')
        ax.vlines(2.5, 0, 1.05, color='grey', linestyle=':', lw=0.8, alpha=0.7)

    plt.suptitle('CNN — Best config per scaler  (◆ = best seed/arch/channel, grey boxes = full distribution)\n'
                 '* CNN best config > FS ROI on all folds (sign test p < 0.05, one-sided)',
                 fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.savefig(root_dir / 'brainage_agg/outputs/figures/classif_cnn_best_per_scaler.png',
                dpi=150, bbox_inches='tight')
    plt.show()

    # Print the winning configs
    print('\nBest CNN config per scaler × aggregation (LogisticReg):')
    lr_best = best_configs[best_configs['classifier']=='LogisticReg'][
        ['scaler','aggregation','cnn_arch','channels','cnn_seed','best_bacc']
    ]
    lr_best = lr_best[lr_best['aggregation'].isin(aggs_shown)].sort_values(['aggregation','scaler'])
    print(lr_best.to_string(index=False))

In [ ]:
# ── §15b Figure 4: Best CNN config per scaler — AUC-ROC ─────────────────────
# Mirrors Figure 3 above but for AUC instead of balanced accuracy.
if len(cnn_cl) > 0:
    _auc_per_file = (cnn_cl
                     .groupby(['cnn_arch','scaler','channels','cnn_seed','aggregation','classifier'])
                     [['bacc','auc']].mean().reset_index())
    _auc_scalers      = ['minmax', 'zscore', 'robust']
    _auc_scaler_colors = {'minmax': '#EF5350', 'zscore': '#FFA726', 'robust': '#66BB6A'}
    _auc_aggs = ['cross_sectional','mean','concatenation','annualized_rate','difference','lme_slope_change']
    x_a = np.arange(len(_auc_aggs))
    w_a = 0.25

    # Seed-averaged per (arch, scaler, channels, agg, clf) then best config per (scaler, agg, clf)
    _sa_auc = (_auc_per_file
               .groupby(['cnn_arch','scaler','channels','aggregation','classifier'])
               ['auc'].mean().reset_index())
    _idx_auc = _sa_auc.groupby(['scaler','aggregation','classifier'])['auc'].idxmax()
    _best_auc = _sa_auc.loc[_idx_auc][
        ['scaler','aggregation','classifier','cnn_arch','channels','auc']
    ].rename(columns={'auc': 'best_auc'}).reset_index(drop=True)

    # Fold-level AUC for the best config (for sign test)
    _best_auc_folds = {}
    for _, _bc in _best_auc.iterrows():
        _k = (_bc['scaler'], _bc['aggregation'], _bc['classifier'])
        _fc = cnn_cl[
            (cnn_cl['cnn_arch']    == _bc['cnn_arch'])   &
            (cnn_cl['scaler']      == _bc['scaler'])      &
            (cnn_cl['channels']    == _bc['channels'])    &
            (cnn_cl['aggregation'] == _bc['aggregation']) &
            (cnn_cl['classifier']  == _bc['classifier'])
        ]
        _best_auc_folds[_k] = _fc.sort_values('fold')['auc'].values

    _fs_auc_summary = (fs_df.groupby(['aggregation','classifier'])[['bacc','auc']]
                       .agg(['mean','std']).reset_index())
    _fs_auc_summary.columns = ['aggregation','classifier','bacc_mean','bacc_std','auc_mean','auc_std']

    fig, axes = plt.subplots(1, 2, figsize=(16, 5), sharey=True)
    for ax, clf in zip(axes, ['LogisticReg', 'RandomForest']):
        sub_dist = _auc_per_file[_auc_per_file['classifier'] == clf]
        sub_best = _best_auc[_best_auc['classifier'] == clf]
        _fs_folds_auc = {
            a: fs_df[(fs_df['aggregation'] == a) & (fs_df['classifier'] == clf)]
                     .sort_values('fold')['auc'].values
            for a in _auc_aggs
        }

        for i, scaler in enumerate(_auc_scalers):
            positions = x_a + (i - 1) * w_a
            data = [
                sub_dist[(sub_dist['scaler'] == scaler) & (sub_dist['aggregation'] == a)]['auc'].values
                for a in _auc_aggs
            ]
            bp = ax.boxplot(data, positions=positions, widths=w_a * 0.8,
                            patch_artist=True, notch=False,
                            medianprops={'color': 'grey', 'lw': 1},
                            whiskerprops={'lw': 0.8, 'color': 'grey'},
                            capprops={'lw': 0.8, 'color': 'grey'},
                            flierprops={'marker': '.', 'markersize': 2, 'alpha': 0.2, 'color': 'grey'})
            for patch in bp['boxes']:
                patch.set_facecolor('#EEEEEE'); patch.set_alpha(0.5)

            s_best = sub_best[sub_best['scaler'] == scaler]
            for xi, a in enumerate(_auc_aggs):
                row = s_best[s_best['aggregation'] == a]
                if len(row):
                    bv = row['best_auc'].values[0]
                    ax.scatter(xi + (i - 1) * w_a, bv, marker='D', s=60,
                               color=_auc_scaler_colors[scaler], zorder=5,
                               edgecolors='black', linewidths=0.5)
                    ax.text(xi + (i - 1) * w_a, bv + 0.008, f'{bv:.3f}',
                            ha='center', va='bottom', fontsize=6,
                            color=_auc_scaler_colors[scaler], fontweight='bold')
                    _cnn_f = _best_auc_folds.get((scaler, a, clf), np.array([]))
                    _fs_f  = _fs_folds_auc.get(a, np.array([]))
                    if len(_cnn_f) > 0 and len(_fs_f) == len(_cnn_f) and bv > _fs_f.mean():
                        _n_better = int(np.sum(_cnn_f > _fs_f))
                        _p_sign = scipy_stats.binom.sf(_n_better - 1, len(_cnn_f), 0.5)
                        if _p_sign < 0.05:
                            ax.text(xi + (i - 1) * w_a, bv + 0.022, '*',
                                    ha='center', va='bottom', fontsize=14,
                                    color='black', fontweight='bold')

        # FS ROI reference dotted lines
        for xi, a in enumerate(_auc_aggs):
            fv = _fs_auc_summary[(_fs_auc_summary['classifier'] == clf) &
                                  (_fs_auc_summary['aggregation'] == a)]['auc_mean'].values
            if len(fv):
                ax.hlines(fv[0], xi - 0.45, xi + 0.45, color='black', lw=2,
                          linestyle=':', label='FS ROI' if xi == 0 else '')

        ax.axhline(0.5, color='grey', linestyle='--', lw=1, alpha=0.6)
        ax.set_xticks(x_a)
        ax.set_xticklabels([AGG_LABELS_CL.get(a, a) for a in _auc_aggs])
        ax.set(ylabel='AUC-ROC' if clf == 'LogisticReg' else '',
               title=clf, ylim=(0.4, 1.05))
        ax.vlines(2.5, 0, 1.05, color='grey', linestyle=':', lw=0.8, alpha=0.7)

        handles = (
            [mpatches.Patch(facecolor=_auc_scaler_colors[s], alpha=0.9, label=s)
             for s in _auc_scalers] +
            [plt.Line2D([0], [0], marker='D', color='w', markerfacecolor='grey',
                        markeredgecolor='black', markersize=7, label='Best config'),
             plt.Line2D([0], [0], color='black', lw=2, linestyle=':', label='FS ROI')]
        )
        ax.legend(handles=handles, fontsize=9, loc='lower left')

    plt.suptitle('CNN — Best config per scaler — AUC-ROC\n'
                 '◆ = best arch/channels per scaler×aggregation  |  grey boxes = full distribution\n'
                 '* CNN best config > FS ROI on all folds (sign test p < 0.05, one-sided)',
                 fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.savefig(root_dir / 'brainage_agg/outputs/figures/classif_cnn_best_per_scaler_auc.png',
                dpi=150, bbox_inches='tight')
    plt.show()


### Classification heatmap — best extractor × aggregation × model

In [ ]:
# ── Figure 5: Heatmap — best performance by extractor × aggregation × model ──
AGG_ORDER_HM = ['cross_sectional','mean','concatenation','annualized_rate',
                'difference','lme_slope_change']
AGG_LABELS_HM = {
    'cross_sectional': 'Cross-sect.',
    'mean':            'Mean',
    'concatenation':   'Concat',
    'annualized_rate': 'Ann. Rate',
    'difference':      'Difference',
    'lme_slope_change':'LME Slope',
}

# FS ROI: mean over CV folds
fs_summary = (fs_df.groupby(['aggregation','classifier'])[['bacc','auc']]
              .mean().reset_index())

# CNN: mean over folds per config, then best config per (agg, clf)
cnn_per_config = (cnn_cl.groupby(['cnn_arch','scaler','channels','cnn_seed',
                                   'aggregation','classifier'])[['bacc','auc']]
                  .mean().reset_index())
cnn_best_cfg = (cnn_per_config
                .groupby(['aggregation','classifier'])[['bacc','auc']]
                .max().reset_index())

cols = [
    ('FS ROI\nLogReg',   'fs_roi', 'LogisticReg'),
    ('FS ROI\nRF',       'fs_roi', 'RandomForest'),
    ('CNN best\nLogReg', 'cnn',    'LogisticReg'),
    ('CNN best\nRF',     'cnn',    'RandomForest'),
]

for metric, metric_label in [('bacc', 'Balanced Accuracy'), ('auc', 'AUC-ROC')]:
    pivot = pd.DataFrame(index=[AGG_LABELS_HM[a] for a in AGG_ORDER_HM],
                         columns=[c[0] for c in cols], dtype=float)
    for col_label, ext, clf in cols:
        src_df = fs_summary if ext == 'fs_roi' else cnn_best_cfg
        for agg in AGG_ORDER_HM:
            val = src_df[(src_df['aggregation']==agg)&(src_df['classifier']==clf)][metric].values
            pivot.loc[AGG_LABELS_HM[agg], col_label] = val[0] if len(val) else np.nan

    fig, ax = plt.subplots(figsize=(9, 5))
    import seaborn as sns
    sns.heatmap(
        pivot.astype(float), annot=True, fmt='.3f',
        cmap='RdYlGn', vmin=0.5, vmax=1.0,
        ax=ax, linewidths=0.5, linecolor='white',
        cbar_kws={'label': metric_label, 'shrink': 0.8},
        annot_kws={'size': 11, 'weight': 'bold'},
    )
    # Vertical separator between FS ROI and CNN columns
    ax.axvline(2, color='black', lw=2)

    ax.set_title(
        f'Child vs Adult classification — {metric_label}\n'
        f'FS ROI (mean over folds)  |  CNN best (max over 240 variants, mean-fold)',
        fontweight='bold', fontsize=12,
    )
    ax.set_xlabel(''); ax.set_ylabel('')
    ax.tick_params(axis='x', rotation=0)
    ax.tick_params(axis='y', rotation=0)

    # Column group labels
    ax.text(1,   -0.55, 'FS ROI',   ha='center', va='top', fontsize=10,
            fontweight='bold', transform=ax.get_xaxis_transform())
    ax.text(3,   -0.55, 'CNN best', ha='center', va='top', fontsize=10,
            fontweight='bold', transform=ax.get_xaxis_transform())

    plt.tight_layout()
    plt.savefig(root_dir / f'brainage_agg/outputs/figures/classif_heatmap_{metric}.png',
                dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
# ── Summary table ────────────────────────────────────────────────────────────
print('\n=== Classification summary (FS ROI) ===')
tbl = (fs_df.groupby(['aggregation','classifier'])[['bacc','auc','f1_macro']]
       .mean().round(3).unstack('classifier'))
tbl = tbl.reindex([a for a in AGG_ORDER_CL if a in tbl.index])
print(tbl.to_string())

---
## 15c  Group-difference analysis: Child vs Adult brain age progression

For each **(extractor × aggregation)** cell, two complementary tests compare the
aggregated brain-feature vectors between Child and Adult subjects:

- **Option 2** — Feature-wise Welch t-test + BH-FDR: fraction of elements that
  significantly differ (FDR < 0.05).
- **Option 4** — Permutation MANOVA (Hotelling's T² in PCA space, 1 000 permutations):
  global multivariate separability, reported as −log₁₀(p).
- **Option 3 raw baseline** — per-feature OLS slope t-test on raw observations
  (before aggregation), shown for FS ROI only.

The `cross_sectional` baseline from §15b is the single-timepoint upper bound.

In [ ]:
# ── §15c: Group-difference results ──────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns
from pathlib import Path

gd = pd.read_csv(root_dir / 'brainage_agg/outputs/group_diff/summary.csv')
cl = pd.read_csv(root_dir / 'brainage_agg/outputs/classif_results.csv')

# ── Aggregation display order (matching §15b) ─────────────────────────────
AGG_ORDER = ['cross_sectional', 'mean', 'concatenation',
             'annualized_rate', 'difference', 'lme_slope_change']
AGG_LABELS = {
    'cross_sectional': 'Cross-sect.',
    'mean':            'Mean',
    'concatenation':   'Concat',
    'annualized_rate': 'Ann. Rate',
    'difference':      'Difference',
    'lme_slope':       'LME Slope',
    'lme_slope_change':'LME Slope\n(change)',
}

# Coarse extractor groups for heatmaps
def coarse_group(g):
    if g == 'FS ROI':          return 'FS ROI'
    if g.startswith('cov'):    return 'CNN (cov_pool)'
    if g.startswith('double'): return 'CNN (double_conv)'
    return g

gd['coarse'] = gd['extractor_group'].apply(coarse_group)

# ── Figure A: frac_sig heatmap ────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (metric, label, cmap) in zip(axes, [
    ('frac_sig',         'Fraction significant (FDR<0.05)',  'YlOrRd'),
    ('manova_neg_log_p', '−log₁₀(MANOVA p)',               'YlOrRd'),
]):
    pivot = (
        gd.groupby(['coarse', 'aggregation'])[metric]
        .mean()
        .reset_index()
        .pivot(index='coarse', columns='aggregation', values=metric)
    )
    cols = [c for c in AGG_ORDER if c in pivot.columns]
    pivot = pivot[cols].rename(columns=AGG_LABELS)
    sns.heatmap(pivot, annot=True, fmt='.2f', cmap=cmap, ax=ax,
                linewidths=0.4, linecolor='white',
                cbar_kws={'label': label, 'shrink': 0.8},
                annot_kws={'size': 9})
    ax.set_title(label, fontweight='bold')
    ax.set_xlabel(''); ax.set_ylabel('')
    ax.tick_params(axis='x', rotation=30)
    ax.tick_params(axis='y', rotation=0)

plt.suptitle('Child vs Adult group-difference tests (Option 2 + Option 4)\n'
             'per extractor group × aggregation (mean across seeds)',
             fontsize=12, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(root_dir / 'brainage_agg/outputs/figures/group_diff_heatmaps.png',
            dpi=150, bbox_inches='tight')
plt.show()

# ── Figure B: Classification BACC vs MANOVA significance (scatter) ────────
# Average classif BACC over folds per (extractor_type, aggregation, classifier)
cl_mean = (cl.groupby(['extractor_type', 'aggregation', 'classifier'])['bacc']
             .mean().reset_index())

# Map group_diff coarse groups to extractor_type
gd_mean = (gd.groupby(['coarse', 'aggregation'])[['frac_sig', 'manova_neg_log_p']]
             .mean().reset_index())
gd_mean['extractor_type'] = gd_mean['coarse'].map({
    'FS ROI': 'fs_roi', 'CNN (cov_pool)': 'cnn', 'CNN (double_conv)': 'cnn'})

merged = pd.merge(
    cl_mean, gd_mean,
    on=['extractor_type', 'aggregation'], how='inner'
)

fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=False)
clf_styles = {'LogisticReg': ('o', '#1976D2'), 'RandomForest': ('s', '#E64A19')}

for ax, x_col, x_label in zip(axes,
    ['frac_sig',         'manova_neg_log_p'],
    ['Fraction significant (FDR<0.05)', '−log₁₀(MANOVA p)']):

    for clf, (marker, color) in clf_styles.items():
        sub = merged[merged['classifier'] == clf]
        ax.scatter(sub[x_col], sub['bacc'],
                   marker=marker, color=color, alpha=0.7, s=60, label=clf)
        # Label aggregations for FS ROI
        for _, row in sub[sub['extractor_type'] == 'fs_roi'].iterrows():
            ax.annotate(AGG_LABELS.get(row['aggregation'], row['aggregation']),
                        (row[x_col], row['bacc']),
                        fontsize=7, ha='left', va='bottom',
                        xytext=(3, 3), textcoords='offset points')

    ax.axhline(0.5, color='grey', linestyle='--', linewidth=0.8, label='Chance')
    ax.set_xlabel(x_label)
    ax.set_ylabel('Balanced Accuracy (Child vs Adult)')
    ax.set_title(f'Classif. BACC vs {x_label}')
    ax.legend(fontsize=8)
    ax.set_ylim(0.45, 1.05)

plt.suptitle('Classification performance vs statistical group-difference strength',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(root_dir / 'brainage_agg/outputs/figures/group_diff_vs_classif.png',
            dpi=150, bbox_inches='tight')
plt.show()

# ── Figure C: FS ROI volcano plots (Options 2 & 3) ────────────────────────
from IPython.display import Image, display
import os

fig_dir = root_dir / 'brainage_agg/outputs/group_diff'
agg_figs = [
    ('option3_raw_baseline', 'Option 3 — raw OLS slope baseline (before aggregation)'),
    ('mean',             'Option 2 — Mean aggregation'),
    ('difference',       'Option 2 — Difference aggregation'),
    ('annualized_rate',  'Option 2 — Annualized rate'),
    ('lme_slope',        'Option 2 — LME slope'),
    ('lme_slope_change', 'Option 2 — LME slope change'),
    ('concatenation',    'Option 2 — Concatenation'),
]
for stem, title in agg_figs:
    path = fig_dir / f'volcano_fs_roi_{stem}.png'
    if path.exists():
        print(f'\n{title}')
        display(Image(str(path), width=2*650))

# ── Table: FS ROI group-diff summary ─────────────────────────────────────
print('\n=== FS ROI group-difference summary ===')
fs_gd = gd[gd['extractor_group'] == 'FS ROI'][
    ['aggregation', 'n_total_elements', 'n_significant', 'frac_sig',
     'manova_t2', 'manova_p', 'n_child', 'n_adult']
].set_index('aggregation').reindex(
    [a for a in ['mean','concatenation','annualized_rate','lme_slope',
                 'difference','lme_slope_change'] if a in gd['aggregation'].values]
).round(3)
print(fs_gd.to_string())


## 16  Summary

In [ ]:
chance = df[(df['target_arm']=='B')&(df['age_band']=='all')]['chance_mae'].mean()

def gmae(arm, agg, ext=None):
    sub = df[(df['target_arm']==arm)&(df['aggregation']==agg)&(df['age_band']=='all')]
    if ext: sub = sub[sub['extractor_type']==ext]
    return sub['MAE'].mean(), sub['MAE'].std()

print('='*70)
print('SUMMARY OF FINDINGS')
print('='*70)

print('\n── Arm A: Absolute age prediction ──')
for agg in AGG_ORDER_A:
    m_fs, _  = gmae('A', agg, 'fs_roi')
    m_cnn, s = gmae('A', agg, 'cnn')
    tag = ' ← STATE' if agg in ['mean','concatenation'] else ' ← change'
    print(f'  {agg:22s}  FS={m_fs:.2f}  CNN={m_cnn:.2f}±{s:.2f}{tag}')

print(f'\n── Arm B: Brain change rate  (chance MAE = {chance:.3f}) ──')
for agg in AGG_ORDER_B:
    m_fs, _  = gmae('B', agg, 'fs_roi')
    m_cnn, s = gmae('B', agg, 'cnn')
    tag = ' ← CHANGE' if agg in ['annualized_rate','difference','lme_slope_change'] else ' ← NEG CTRL'
    print(f'  {agg:22s}  FS={m_fs:.3f}  CNN={m_cnn:.3f}±{s:.3f}{tag}')

print('\n── CNN variant spread (Arm A mean agg) ──')
sub_m = cnn_per_file[(cnn_per_file['target_arm']=='A')&(cnn_per_file['aggregation']=='mean')]
print(f'  Across all 120 variants: [{sub_m["MAE"].min():.2f}, {sub_m["MAE"].max():.2f}] yrs  '
      f'(range {sub_m["MAE"].max()-sub_m["MAE"].min():.2f})')
for scaler, c in SCALER_COLORS.items():
    vals = sub_m[sub_m['scaler']==scaler]['MAE']
    print(f'  scaler={scaler:8s}: {vals.mean():.3f} ± {vals.std():.3f}  [{vals.min():.2f}, {vals.max():.2f}]')

print('\n── CNN variant spread (Arm B difference agg) ──')
sub_d = cnn_per_file[(cnn_per_file['target_arm']=='B')&(cnn_per_file['aggregation']=='difference')]
print(f'  Across all 120 variants: [{sub_d["MAE"].min():.3f}, {sub_d["MAE"].max():.3f}] yrs')
for scaler, c in SCALER_COLORS.items():
    vals = sub_d[sub_d['scaler']==scaler]['MAE']
    print(f'  scaler={scaler:8s}: {vals.mean():.3f} ± {vals.std():.3f}  [{vals.min():.3f}, {vals.max():.3f}]')

---
## 16  G1 vs G2 — Span Classification

**Hypothesis:** If the brain trajectory is monotone in feature space, the full-span
difference `f_last − f_first` (G1) is distinguishable from the short-span difference
`f_mid − f_first` (G2).  Near-chance AUC ≈ 0.5 implies a linear trajectory
(G1 ≈ scaled version of G2); AUC >> 0.5 implies non-linearity or accelerating change.

Input features: raw difference vectors. GroupKFold on subject (both G1 and G2 samples
of the same subject always in the same fold).  Run per band to avoid Child/Adult
age confounds.

Run the SLURM array first:
```
sbatch brainage_agg/slurm/submit_span_classif.sh
python brainage_agg/experiment/run_span_classification.py --merge
```


In [ ]:
_span_path = root_dir / 'brainage_agg/outputs/span_classif_results.csv'

if not _span_path.exists():
    print("span_classif_results.csv not found.")
    print("Run:  sbatch brainage_agg/slurm/submit_span_classif.sh")
    print("Then: python brainage_agg/experiment/run_span_classification.py --merge")
else:
    span_df = pd.read_csv(_span_path)
    if 'variant' not in span_df.columns:
        span_df['variant'] = 'difference'
    span_df = span_df[span_df['variant'] == 'difference'].copy()
    print(f"Loaded {len(span_df)} rows (difference variant) | extractors: {span_df['extractor'].nunique()}")

    BANDS_SPAN   = ['Child', 'Adult']
    CLFS_SPAN    = ['LogisticReg', 'RandomForest']
    CLF_LABELS_S = {'LogisticReg': 'Logistic Reg.', 'RandomForest': 'Random Forest'}
    EXT_COLORS   = {'fs_roi': '#2196F3', 'cnn': '#FF9800'}
    EXT_LABELS   = {'fs_roi': 'FS ROI', 'cnn': 'CNN (best config)'}

    def _best_cnn_fold_vals(band, clf, metric):
        """Fold-level values for the best CNN config (seeds avg within fold)."""
        sub = span_df[
            (span_df['extractor_type'] == 'cnn') &
            (span_df['band'] == band) &
            (span_df['classifier'] == clf)
        ]
        per_seed = (sub.groupby(['cnn_arch', 'scaler', 'channels', 'cnn_seed'])[metric]
                    .mean().reset_index())
        cfg_mean = per_seed.groupby(['cnn_arch', 'scaler', 'channels'])[metric].mean()
        best_arch, best_scaler, best_ch = cfg_mean.idxmax()
        return (sub[(sub['cnn_arch'] == best_arch) &
                    (sub['scaler']   == best_scaler) &
                    (sub['channels'] == best_ch)]
                .groupby('fold')[metric].mean())

    def _ci95(fold_vals):
        n = len(fold_vals)
        if n < 1:
            return np.nan, 0.0
        return float(fold_vals.mean()), float(fold_vals.std())

    def _span_stats_fs(band, clf, metric):
        """FS ROI: mean ± 95% CI over 5 folds."""
        by_fold = (span_df[(span_df['extractor_type'] == 'fs_roi') &
                           (span_df['band'] == band) &
                           (span_df['classifier'] == clf)]
                   .groupby('fold')[metric].mean())
        return _ci95(by_fold)

    def _span_stats_best_cnn(band, clf, metric):
        """Best CNN config: mean ± 95% CI over 5 folds (seeds avg within fold)."""
        return _ci95(_best_cnn_fold_vals(band, clf, metric))

    def _span_pval(band, clf, metric):
        """Wilcoxon signed-rank: H1 CNN > FS ROI, paired by fold (n=5)."""
        fs_fold  = (span_df[(span_df['extractor_type'] == 'fs_roi') &
                            (span_df['band'] == band) &
                            (span_df['classifier'] == clf)]
                    .groupby('fold')[metric].mean())
        cnn_fold = _best_cnn_fold_vals(band, clf, metric)
        m = pd.DataFrame({'fs': fs_fold, 'cnn': cnn_fold}).dropna()
        if len(m) < 3:
            return float('nan')
        try:
            return wilcoxon(m['cnn'].values, m['fs'].values,
                            alternative='greater').pvalue
        except Exception:
            return float('nan')

    metrics_span = [('bacc', 'Balanced Accuracy'), ('auc', 'AUC-ROC')]
    ext_order    = ['fs_roi', 'cnn']

    fig, axes = plt.subplots(2, 2, figsize=(13, 9), sharey='row')

    for row, (metric, metric_label) in enumerate(metrics_span):
        for col, clf in enumerate(CLFS_SPAN):
            ax = axes[row][col]
            x  = np.arange(len(BANDS_SPAN))
            w  = 0.8 / len(ext_order)

            # Store per-band top-of-bar positions for bracket drawing
            bar_tops = {ext: [] for ext in ext_order}

            for i, ext_type in enumerate(ext_order):
                means, errs = [], []
                for band in BANDS_SPAN:
                    if ext_type == 'fs_roi':
                        m, e = _span_stats_fs(band, clf, metric)
                    else:
                        m, e = _span_stats_best_cnn(band, clf, metric)
                    means.append(m); errs.append(e)
                    bar_tops[ext_type].append(m + e)

                bars = ax.bar(
                    x + (i - len(ext_order)/2 + 0.5) * w, means, w,
                    label=EXT_LABELS[ext_type], color=EXT_COLORS[ext_type],
                    alpha=0.85, yerr=errs, capsize=5,
                    error_kw={'lw': 1.5, 'capthick': 1.5},
                )
                for bar, m, e in zip(bars, means, errs):
                    if not np.isnan(m):
                        ax.text(bar.get_x() + bar.get_width() / 2,
                                m + e + 0.005, f'{m:.3f}',
                                ha='center', va='bottom', fontsize=8, fontweight='bold')

            # Significance brackets
            for xi, band in enumerate(BANDS_SPAN):
                p   = _span_pval(band, clf, metric)
                sig = '**' if p < 0.01 else '*' if p < 0.05 else 'ns'
                if sig == 'ns':
                    continue
                y_top  = max(bar_tops['fs_roi'][xi], bar_tops['cnn'][xi]) + 0.1
                x_fs   = x[xi] + (0 - len(ext_order)/2 + 0.5) * w
                x_cnn  = x[xi] + (1 - len(ext_order)/2 + 0.5) * w
                tick_h = 0.008
                ax.plot([x_fs,  x_fs,  x_cnn, x_cnn],
                        [y_top - tick_h, y_top, y_top, y_top - tick_h],
                        color='black', lw=1.2)
                ax.text((x_fs + x_cnn) / 2, y_top + 0.003, sig,
                        ha='center', va='bottom', fontsize=12, fontweight='bold')

            ax.axhline(0.5, color='grey', linestyle='--', lw=1.2, alpha=0.7,
                       label='Chance (0.5)')
            ax.set_xticks(x)
            ax.set_xticklabels(BANDS_SPAN, fontsize=11)
            ax.set(ylabel=metric_label if col == 0 else '',
                   title=CLF_LABELS_S[clf], ylim=(0.3, 1.05))
            if row == 0 and col == 0:
                ax.legend(fontsize=9, loc='upper right')

        plt.setp(axes[row][1].get_yticklabels(), visible=False)

    for row, (_, metric_label) in enumerate(metrics_span):
        axes[row][0].set_ylabel(metric_label, fontsize=11)

    plt.suptitle(
        'G1 (full span) vs G2 (short span) — Span Classification\n'
        'mean ± std over 5 folds  (* p<0.05, Wilcoxon, CNN vs FS ROI)',
        fontsize=12, fontweight='bold'
    )
    plt.tight_layout()
    plt.savefig(root_dir / 'brainage_agg/outputs/figures/span_classif_summary.png',
                dpi=150, bbox_inches='tight')
    plt.show()


### 16b  CNN vs FS ROI — statistical test + variant bar plots (span classification)

In [ ]:
# ── §16b  CNN vs FS ROI: stat test + bar plots for span classification ─────

_span_path = root_dir / 'brainage_agg/outputs/span_classif_results.csv'

if not _span_path.exists():
    print('span_classif_results.csv not found — run submit_span_classif.sh first.')
else:
    _sp = pd.read_csv(_span_path)
    if 'variant' not in _sp.columns: _sp['variant'] = 'difference'
    _sp = _sp[_sp['variant'] == 'difference'].copy()

    # --- Statistical test: is CNN bacc > FS ROI bacc? (paired by fold, n=5) ---
    print('Wilcoxon signed-rank: H₁  CNN bacc > FS bacc  (paired by CV fold, n=5)')
    print(f"{'Band':<8} {'Classifier':<14} {'FS bacc':>8} {'CNN bacc':>9} {'Δbacc':>7} {'p':>10}  sig")
    print('─' * 72)

    _sig_rows = []

    for _band in ['Child', 'Adult']:
        for _clf in ['LogisticReg', 'RandomForest']:
            _sub = _sp[(_sp['band']==_band) & (_sp['classifier']==_clf)]
            _fs  = _sub[_sub['extractor_type']=='fs_roi'].groupby('fold')['bacc'].mean()
            for _cnn_arch in _sub[_sub['extractor_type']=='cnn']['cnn_arch'].unique():
                for _scaler in _sub[(_sub['extractor_type']=='cnn') & (_sub['cnn_arch']==_cnn_arch)]['scaler'].unique():
                    for _channel in _sub[(_sub['extractor_type']=='cnn') & (_sub['cnn_arch']==_cnn_arch) & (_sub['scaler']==_scaler)]['channels'].unique():
                        _cnn = _sub[(_sub['extractor_type']=='cnn') & (_sub['cnn_arch']==_cnn_arch) & (_sub['scaler']==_scaler) & (_sub['channels']==_channel)].groupby('fold')['bacc'].mean()
                        _m   = pd.DataFrame({'fs': _fs, 'cnn': _cnn}).dropna()
                        if len(_m) < 3:
                            continue
                        try:
                            _p = wilcoxon(_m['cnn'].values, _m['fs'].values, alternative='greater').pvalue
                        except Exception:
                            _p = float('nan')
                        _delta = (_m['cnn'] - _m['fs']).mean()
                        _sig   = '***' if _p < 0.001 else '**' if _p < 0.01 else '*' if _p < 0.05 else 'ns'
                        _sig_rows.append({
                            'band': _band,
                            'classifier': _clf,
                            'cnn_arch': _cnn_arch,
                            'scaler': _scaler,
                            'channels': _channel,
                            'p': _p,
                            'delta_bacc': _delta,
                            'sig': _sig
                        })
    _sig_map = pd.DataFrame(
        _sig_rows,
        columns=['band', 'classifier', 'cnn_arch', 'scaler', 'channels', 'p', 'delta_bacc', 'sig']
    )
    display(_sig_map)
    print('Note: n=5 folds → min achievable one-sided p ≈ 0.063.',
          'Δbacc > 0 means CNN trending better.')

    # --- Bar plots: all 24 CNN configs per band × classifier (bacc) ------------
    _CHANNEL_SHORT_SP = {'t1': 't1', 't1_sobel': 'sob',
                         't1_rank_sobel': 'rnk', 't1_median_sobel': 'med'}

    _cnn_sp = _sp[_sp['extractor_type']=='cnn'].copy()
    _cnn_sp['config'] = (_cnn_sp['cnn_arch'].str[:3] + '/' + _cnn_sp['scaler'] + '/'
                         + _cnn_sp['channels'].map(_CHANNEL_SHORT_SP))
    # avg seeds + folds → 1 bacc per (config, band, classifier)
    _cfg_sp = (_cnn_sp.groupby(['config', 'scaler', 'band', 'classifier'])['bacc']
               .mean().reset_index())

    fig, axes = plt.subplots(2, 2, figsize=(20, 10), sharey=True)

    for _row, _band in enumerate(['Child', 'Adult']):
        for _col, _clf in enumerate(['LogisticReg', 'RandomForest']):
            ax = axes[_row][_col]

            _fs_bacc = (_sp[(_sp['extractor_type']=='fs_roi') &
                            (_sp['band']==_band) & (_sp['classifier']==_clf)]
                        .groupby('fold')['bacc'].mean().mean())

            _cfg_b = (_cfg_sp[(_cfg_sp['band']==_band) & (_cfg_sp['classifier']==_clf)]
                      .sort_values('bacc', ascending=False).reset_index(drop=True))

            _best_cfg = _cfg_b.iloc[0]['config']
            _colors = [SCALER_COLORS.get(s, '#90CAF9')
                       for c, s in zip(_cfg_b['config'], _cfg_b['scaler'])]

            ax.bar(range(len(_cfg_b)), _cfg_b['bacc'], color=_colors,
                   width=0.7, alpha=0.8, zorder=2, edgecolor='none')
            ax.bar(0, _cfg_b.iloc[0]['bacc'], color=_colors[0],
                   width=0.7, alpha=1.0, zorder=3, edgecolor='black', linewidth=1.8)
            ax.text(0, _cfg_b.iloc[0]['bacc'] + 0.005,
                    f'★ {_best_cfg}\n{_cfg_b.iloc[0]["bacc"]:.3f}',
                    ha='center', va='bottom', fontsize=7.5, fontweight='bold',
                    color='#BF360C', zorder=5)

            ax.axhline(_fs_bacc, color='#1565C0', lw=2.5, ls='--', zorder=4)
            ax.axhline(0.5, color='gray', lw=1.2, ls=':', alpha=0.6, zorder=1)
            ax.text(len(_cfg_b) - 1, _fs_bacc + 0.005,
                    f'FS ROI  {_fs_bacc:.3f}',
                    ha='right', va='bottom', fontsize=8.5, color='#1565C0', fontweight='bold')

            ax.set_xticks(range(len(_cfg_b)))
            ax.set_xticklabels(_cfg_b['config'], rotation=45, ha='right', fontsize=7.5)
            ax.set_ylabel('Balanced Accuracy')
            ax.set_title(f'{_band} — {_clf}', fontweight='bold', fontsize=11)
            ax.set_ylim(bottom=0.4, top=1.05)
            ax.grid(axis='y', alpha=0.35, zorder=0)

            _patches = [mpatches.Patch(color=SCALER_COLORS[s], label=s, alpha=0.85)
                        for s in ['minmax', 'zscore', 'robust']]
            _best_p = mpatches.Patch(color="#FFFFFF", label='best config',
                                     linewidth=1.5, edgecolor='black')
            _fs_l = plt.Line2D([0], [0], color='#1565C0', lw=2.5, ls='--',
                               label='FS ROI baseline')
            _ch_l = plt.Line2D([0], [0], color='gray', lw=1.2, ls=':',
                               label='Chance (0.5)', alpha=0.6)
            ax.legend(handles=_patches + [_best_p, _fs_l, _ch_l],
                      fontsize=7.5, loc='upper right', ncol=2)

    plt.suptitle(
        'All 24 CNN configs vs FS ROI — G1 vs G2 span classification (bacc)\n'
        '(avg seeds × folds; bars colored by scaler; ★ = best CNN; dashed = FS ROI)',
        fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.savefig(root_dir / 'brainage_agg/outputs/figures/span_cnn_variants_vs_fs.png',
                dpi=150, bbox_inches='tight')
    plt.show()


In [ ]:
# ── §16b  CNN vs FS ROI: stat test + bar plots for span classification ─────

_span_path = root_dir / 'brainage_agg/outputs/span_classif_results.csv'

if not _span_path.exists():
    print('span_classif_results.csv not found — run submit_span_classif.sh first.')
else:
    _sp = pd.read_csv(_span_path)
    if 'variant' not in _sp.columns: _sp['variant'] = 'difference'
    _sp = _sp[_sp['variant'] == 'difference'].copy()

    # --- Statistical test: is CNN auc > FS ROI auc? (paired by fold, n=5) ---
    print('Wilcoxon signed-rank: H₁  CNN auc > FS auc  (paired by CV fold, n=5)')
    print(f"{'Band':<8} {'Classifier':<14} {'FS auc':>8} {'CNN auc':>9} {'Δauc':>7} {'p':>10}  sig")
    print('─' * 72)

    _sig_rows = []

    for _band in ['Child', 'Adult']:
        for _clf in ['LogisticReg', 'RandomForest']:
            _sub = _sp[(_sp['band']==_band) & (_sp['classifier']==_clf)]
            _fs  = _sub[_sub['extractor_type']=='fs_roi'].groupby('fold')['auc'].mean()
            for _cnn_arch in _sub[_sub['extractor_type']=='cnn']['cnn_arch'].unique():
                for _scaler in _sub[(_sub['extractor_type']=='cnn') & (_sub['cnn_arch']==_cnn_arch)]['scaler'].unique():
                    for _channel in _sub[(_sub['extractor_type']=='cnn') & (_sub['cnn_arch']==_cnn_arch) & (_sub['scaler']==_scaler)]['channels'].unique():
                        _cnn = _sub[(_sub['extractor_type']=='cnn') & (_sub['cnn_arch']==_cnn_arch) & (_sub['scaler']==_scaler) & (_sub['channels']==_channel)].groupby('fold')['auc'].mean()
                        _m   = pd.DataFrame({'fs': _fs, 'cnn': _cnn}).dropna()
                        if len(_m) < 3:
                            continue
                        try:
                            _p = wilcoxon(_m['cnn'].values, _m['fs'].values, alternative='greater').pvalue
                        except Exception:
                            _p = float('nan')
                        _delta = (_m['cnn'] - _m['fs']).mean()
                        _sig   = '***' if _p < 0.001 else '**' if _p < 0.01 else '*' if _p < 0.05 else 'ns'
                        _sig_rows.append({
                            'band': _band,
                            'classifier': _clf,
                            'cnn_arch': _cnn_arch,
                            'scaler': _scaler,
                            'channels': _channel,
                            'p': _p,
                            'delta_auc': _delta,
                            'sig': _sig
                        })
    _sig_map = pd.DataFrame(
        _sig_rows,
        columns=['band', 'classifier', 'cnn_arch', 'scaler', 'channels', 'p', 'delta_auc', 'sig']
    )
    display(_sig_map)
    print('Note: n=5 folds → min achievable one-sided p ≈ 0.063.',
          'Δauc > 0 means CNN trending better.')

    # --- Bar plots: all 24 CNN configs per band × classifier (auc) ------------
    _CHANNEL_SHORT_SP = {'t1': 't1', 't1_sobel': 'sob',
                         't1_rank_sobel': 'rnk', 't1_median_sobel': 'med'}

    _cnn_sp = _sp[_sp['extractor_type']=='cnn'].copy()
    _cnn_sp['config'] = (_cnn_sp['cnn_arch'].str[:3] + '/' + _cnn_sp['scaler'] + '/'
                         + _cnn_sp['channels'].map(_CHANNEL_SHORT_SP))
    # avg seeds + folds → 1 auc per (config, band, classifier)
    _cfg_sp = (_cnn_sp.groupby(['config', 'scaler', 'band', 'classifier'])['auc']
               .mean().reset_index())

    fig, axes = plt.subplots(2, 2, figsize=(20, 10), sharey=True)

    for _row, _band in enumerate(['Child', 'Adult']):
        for _col, _clf in enumerate(['LogisticReg', 'RandomForest']):
            ax = axes[_row][_col]

            _fs_auc = (_sp[(_sp['extractor_type']=='fs_roi') &
                            (_sp['band']==_band) & (_sp['classifier']==_clf)]
                        .groupby('fold')['auc'].mean().mean())

            _cfg_b = (_cfg_sp[(_cfg_sp['band']==_band) & (_cfg_sp['classifier']==_clf)]
                      .sort_values('auc', ascending=False).reset_index(drop=True))

            _best_cfg = _cfg_b.iloc[0]['config']
            _colors = [SCALER_COLORS.get(s, '#90CAF9')
                       for c, s in zip(_cfg_b['config'], _cfg_b['scaler'])]

            ax.bar(range(len(_cfg_b)), _cfg_b['auc'], color=_colors,
                   width=0.7, alpha=0.8, zorder=2, edgecolor='none')
            ax.bar(0, _cfg_b.iloc[0]['auc'], color=_colors[0],
                   width=0.7, alpha=1.0, zorder=3, edgecolor='black', linewidth=1.8)
            ax.text(0, _cfg_b.iloc[0]['auc'] + 0.005,
                    f'★ {_best_cfg}\n{_cfg_b.iloc[0]["auc"]:.3f}',
                    ha='center', va='bottom', fontsize=7.5, fontweight='bold',
                    color='#BF360C', zorder=5)

            ax.axhline(_fs_auc, color='#1565C0', lw=2.5, ls='--', zorder=4)
            ax.axhline(0.5, color='gray', lw=1.2, ls=':', alpha=0.6, zorder=1)
            ax.text(len(_cfg_b) - 1, _fs_auc + 0.005,
                    f'FS ROI  {_fs_auc:.3f}',
                    ha='right', va='bottom', fontsize=8.5, color='#1565C0', fontweight='bold')

            ax.set_xticks(range(len(_cfg_b)))
            ax.set_xticklabels(_cfg_b['config'], rotation=45, ha='right', fontsize=7.5)
            ax.set_ylabel('AUC-ROC')
            ax.set_title(f'{_band} — {_clf}', fontweight='bold', fontsize=11)
            ax.set_ylim(bottom=0.4, top=1.05)
            ax.grid(axis='y', alpha=0.35, zorder=0)

            _patches = [mpatches.Patch(color=SCALER_COLORS[s], label=s, alpha=0.85)
                        for s in ['minmax', 'zscore', 'robust']]
            _best_p = mpatches.Patch(color="#FFFFFF", label='best config',
                                     linewidth=1.5, edgecolor='black')
            _fs_l = plt.Line2D([0], [0], color='#1565C0', lw=2.5, ls='--',
                               label='FS ROI baseline')
            _ch_l = plt.Line2D([0], [0], color='gray', lw=1.2, ls=':',
                               label='Chance (0.5)', alpha=0.6)
            ax.legend(handles=_patches + [_best_p, _fs_l, _ch_l],
                      fontsize=7.5, loc='lower right', ncol=2)

    plt.suptitle(
        'All 24 CNN configs vs FS ROI — G1 vs G2 span classification (auc)\n'
        '(avg seeds × folds; bars colored by scaler; ★ = best CNN; dashed = FS ROI)',
        fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.savefig(root_dir / 'brainage_agg/outputs/figures/span_cnn_variants_vs_fs_auc.png',
                dpi=150, bbox_inches='tight')
    plt.show()


In [ ]:
# ── §16c  CNN — best config per scaler (span classif, same style as §15b Fig 4) ─
_span_path = root_dir / 'brainage_agg/outputs/span_classif_results.csv'

if not _span_path.exists():
    print('span_classif_results.csv not found — run submit_span_classif.sh first.')
else:
    _sp2 = pd.read_csv(_span_path)
    if 'variant' not in _sp2.columns: _sp2['variant'] = 'difference'
    _sp2 = _sp2[_sp2['variant'] == 'difference'].copy()
    _bands_shown = ['Child', 'Adult']
    _scalers_sp  = ['minmax', 'zscore', 'robust']
    _scaler_col  = {'minmax': '#EF5350', 'zscore': '#FFA726', 'robust': '#66BB6A'}
    _metrics     = [('bacc', 'Balanced Accuracy'), ('auc', 'AUC-ROC')]

    # Per-file fold-average for both metrics
    _cnn_sp_pf = (_sp2[_sp2['extractor_type']=='cnn']
                  .groupby(['cnn_arch','scaler','channels','cnn_seed','band','classifier'])
                  [['bacc','auc']].mean().reset_index())
    _fs_sp2 = (_sp2[_sp2['extractor_type']=='fs_roi']
               .groupby(['band','classifier'])[['bacc','auc']].mean().reset_index())

    # ── Pre-compute Wilcoxon p-values for best config per scaler vs FS ROI ────
    # Fold-level data: avg seeds per (arch, scaler, channels, band, clf, fold)
    _cnn_bycfg = (_sp2[_sp2['extractor_type']=='cnn']
                  .groupby(['cnn_arch','scaler','channels','band','classifier','fold'])
                  [['bacc','auc']].mean().reset_index())

    _pval_cache = {}  # (metric, scaler, band, clf) → p-value
    for _met, _ in _metrics:
        for _sc in _scalers_sp:
            for _bd in _bands_shown:
                for _cl in ['LogisticReg', 'RandomForest']:
                    _sub = _cnn_bycfg[(_cnn_bycfg['scaler']==_sc) &
                                      (_cnn_bycfg['band']==_bd) &
                                      (_cnn_bycfg['classifier']==_cl)]
                    _cfg_mean = _sub.groupby(['cnn_arch','channels'])[_met].mean()
                    _best_arch, _best_ch = _cfg_mean.idxmax()
                    _cnn_f = (_sub[(_sub['cnn_arch']==_best_arch) &
                                   (_sub['channels']==_best_ch)]
                              .set_index('fold')[_met])
                    _fs_f = (_sp2[(_sp2['extractor_type']=='fs_roi') &
                                  (_sp2['band']==_bd) &
                                  (_sp2['classifier']==_cl)]
                             .groupby('fold')[_met].mean())
                    _m = pd.DataFrame({'cnn': _cnn_f, 'fs': _fs_f}).dropna()
                    try:
                        _p = wilcoxon(_m['cnn'].values, _m['fs'].values,
                                      alternative='greater').pvalue
                    except Exception:
                        _p = float('nan')
                    _pval_cache[(_met, _sc, _bd, _cl)] = _p

    x = np.arange(len(_bands_shown))
    w = 0.25

    fig, axes = plt.subplots(2, 2, figsize=(10, 9), sharey='row')

    for row, (metric, metric_lbl) in enumerate(_metrics):
        _best = (_cnn_sp_pf.groupby(['scaler','band','classifier'])[metric]
                 .max().reset_index().rename(columns={metric: 'best_val'}))

        for col, clf in enumerate(['LogisticReg', 'RandomForest']):
            ax = axes[row][col]

            for i, scaler in enumerate(_scalers_sp):
                positions = x + (i - 1) * w
                data = [_cnn_sp_pf[(_cnn_sp_pf['scaler']==scaler) &
                                    (_cnn_sp_pf['band']==band) &
                                    (_cnn_sp_pf['classifier']==clf)][metric].values
                        for band in _bands_shown]
                bp = ax.boxplot(data, positions=positions, widths=w*0.8,
                                patch_artist=True, notch=False,
                                medianprops={'color': 'grey', 'lw': 1},
                                whiskerprops={'lw': 0.8, 'color': 'grey'},
                                capprops={'lw': 0.8, 'color': 'grey'},
                                flierprops={'marker': '.', 'markersize': 2,
                                            'alpha': 0.2, 'color': 'grey'})
                for patch in bp['boxes']:
                    patch.set_facecolor('#EEEEEE'); patch.set_alpha(0.5)

                for xi, band in enumerate(_bands_shown):
                    r = _best[(_best['scaler']==scaler) &
                              (_best['band']==band) &
                              (_best['classifier']==clf)]
                    if len(r):
                        bv  = r['best_val'].values[0]
                        p   = _pval_cache.get((metric, scaler, band, clf), float('nan'))
                        sig = '**' if p < 0.01 else '*' if p < 0.05 else ''
                        ax.scatter(xi + (i-1)*w, bv, marker='D', s=60,
                                   color=_scaler_col[scaler], zorder=5,
                                   edgecolors='black', linewidths=0.5)
                        ax.text(xi + (i-1)*w, bv + 0.008,
                                f'{bv:.3f}{sig}',
                                ha='center', va='bottom', fontsize=7,
                                color=_scaler_col[scaler], fontweight='bold')

            for xi, band in enumerate(_bands_shown):
                fv = _fs_sp2[(_fs_sp2['band']==band) &
                             (_fs_sp2['classifier']==clf)][metric].values
                if len(fv):
                    ax.hlines(fv[0], xi - 0.45, xi + 0.45, color='black', lw=2,
                              linestyle=':', label='FS ROI' if xi == 0 else '')

            ax.axhline(0.5, color='grey', linestyle='--', lw=1, alpha=0.6)
            ax.set_xticks(x)
            ax.set_xticklabels(_bands_shown, fontsize=11)
            ax.set(ylabel=metric_lbl if col == 0 else '', ylim=(0.4, 1.05))
            if row == 0:
                ax.set_title(clf, fontweight='bold', fontsize=11)

            handles = ([mpatches.Patch(facecolor=_scaler_col[s], alpha=0.9, label=s)
                        for s in _scalers_sp] +
                       [plt.Line2D([0],[0], marker='D', color='w', markerfacecolor='grey',
                                   markeredgecolor='black', markersize=7, label='Best config'),
                        plt.Line2D([0],[0], color='black', lw=2, linestyle=':', label='FS ROI')])
            ax.legend(handles=handles, fontsize=8, loc='upper right', ncol=2)

    plt.suptitle(
        'CNN — Best config per scaler  (◆ = best seed/arch/channel, grey boxes = full distribution)\n'
        'G1 (full span) vs G2 (short span) — span classification  (* p<0.05 vs FS ROI, Wilcoxon n=5)',
        fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.savefig(root_dir / 'brainage_agg/outputs/figures/span_cnn_best_per_scaler.png',
                dpi=150, bbox_inches='tight')
    plt.show()


### 16d  Concatenation span variant — G1 vs G2

Same test as §16/§16b but using **concatenation** features:
- **G1** = `[f_first, f_mid, f_last]` — full-span concatenation
- **G2** = `[f_first, f_mid, f_mid]` — short-span (last slot = mid)

Near-chance (AUC ≈ 0.5) implies the extra time adds no new signal in
absolute feature space. Unlike the difference variant this test is
immune to catastrophic cancellation.

In [ ]:
# ── §16d  Concatenation span variant ─────────────────────────────────────────
_span_path = root_dir / 'brainage_agg/outputs/span_classif_results.csv'

if not _span_path.exists():
    print('span_classif_results.csv not found — run submit_span_classif.sh first.')
elif 'variant' not in pd.read_csv(_span_path, nrows=1).columns:
    print('No concatenation variant found — re-run jobs with the updated script:')
    print('  sbatch brainage_agg/slurm/submit_span_classif.sh')
    print('  python brainage_agg/experiment/run_span_classification.py --merge')
else:
    _sp_c = pd.read_csv(_span_path)
    _sp_c = _sp_c[_sp_c['variant'] == 'concatenation'].copy()
    if _sp_c.empty:
        print('Concatenation variant rows not found — re-run SLURM jobs.')
    else:
        print(f'Loaded {len(_sp_c)} rows (concatenation variant) | '
              f'extractors: {_sp_c["extractor"].nunique()}')

        BANDS_SPAN_C = ['Child', 'Adult']
        CLFS_SPAN_C  = ['LogisticReg', 'RandomForest']
        CLF_LABELS_C = {'LogisticReg': 'Logistic Reg.', 'RandomForest': 'Random Forest'}
        EXT_COLORS_C = {'fs_roi': '#2196F3', 'cnn': '#FF9800'}
        EXT_LABELS_C = {'fs_roi': 'FS ROI', 'cnn': 'CNN (best config)'}

        def _best_cnn_folds_c(band, clf, metric):
            sub = _sp_c[(_sp_c['extractor_type'] == 'cnn') &
                        (_sp_c['band'] == band) &
                        (_sp_c['classifier'] == clf)]
            per_seed = (sub.groupby(['cnn_arch', 'scaler', 'channels', 'cnn_seed'])[metric]
                        .mean().reset_index())
            cfg_mean = per_seed.groupby(['cnn_arch', 'scaler', 'channels'])[metric].mean()
            best_arch, best_scaler, best_ch = cfg_mean.idxmax()
            return (sub[(sub['cnn_arch'] == best_arch) &
                        (sub['scaler']   == best_scaler) &
                        (sub['channels'] == best_ch)]
                    .groupby('fold')[metric].mean())

        def _ci_c(fv):
            return (float(fv.mean()), float(fv.std())) if len(fv) else (np.nan, 0.0)

        def _pval_c(band, clf, metric):
            fs_f = (_sp_c[(_sp_c['extractor_type'] == 'fs_roi') &
                          (_sp_c['band'] == band) &
                          (_sp_c['classifier'] == clf)]
                    .groupby('fold')[metric].mean())
            cn_f = _best_cnn_folds_c(band, clf, metric)
            m = pd.DataFrame({'fs': fs_f, 'cnn': cn_f}).dropna()
            if len(m) < 3: return float('nan')
            try:
                return wilcoxon(m['cnn'].values, m['fs'].values,
                                alternative='greater').pvalue
            except Exception:
                return float('nan')

        metrics_c  = [('bacc', 'Balanced Accuracy'), ('auc', 'AUC-ROC')]
        ext_order_c = ['fs_roi', 'cnn']

        fig, axes = plt.subplots(2, 2, figsize=(13, 9), sharey='row')

        for row, (metric, metric_label) in enumerate(metrics_c):
            for col, clf in enumerate(CLFS_SPAN_C):
                ax = axes[row][col]
                x  = np.arange(len(BANDS_SPAN_C))
                w  = 0.8 / len(ext_order_c)
                bar_tops = {ext: [] for ext in ext_order_c}

                for i, ext_type in enumerate(ext_order_c):
                    means, errs = [], []
                    for band in BANDS_SPAN_C:
                        if ext_type == 'fs_roi':
                            m_val, e = _ci_c(
                                _sp_c[(_sp_c['extractor_type'] == 'fs_roi') &
                                      (_sp_c['band'] == band) &
                                      (_sp_c['classifier'] == clf)]
                                .groupby('fold')[metric].mean())
                        else:
                            m_val, e = _ci_c(_best_cnn_folds_c(band, clf, metric))
                        means.append(m_val); errs.append(e)
                        bar_tops[ext_type].append(m_val + e)

                    bars = ax.bar(
                        x + (i - len(ext_order_c)/2 + 0.5) * w, means, w,
                        label=EXT_LABELS_C[ext_type], color=EXT_COLORS_C[ext_type],
                        alpha=0.85, yerr=errs, capsize=5,
                        error_kw={'lw': 1.5, 'capthick': 1.5},
                    )
                    for bar, m_val, e in zip(bars, means, errs):
                        if not np.isnan(m_val):
                            ax.text(bar.get_x() + bar.get_width() / 2,
                                    m_val + e + 0.005, f'{m_val:.3f}',
                                    ha='center', va='bottom', fontsize=8,
                                    fontweight='bold')

                for xi, band in enumerate(BANDS_SPAN_C):
                    p   = _pval_c(band, clf, metric)
                    sig = '**' if p < 0.01 else '*' if p < 0.05 else 'ns'
                    if sig == 'ns':
                        continue
                    y_top = max(bar_tops['fs_roi'][xi], bar_tops['cnn'][xi]) + 0.1
                    x_fs  = x[xi] + (0 - len(ext_order_c)/2 + 0.5) * w
                    x_cnn = x[xi] + (1 - len(ext_order_c)/2 + 0.5) * w
                    tick_h = 0.008
                    ax.plot([x_fs,  x_fs,  x_cnn, x_cnn],
                            [y_top - tick_h, y_top, y_top, y_top - tick_h],
                            color='black', lw=1.2)
                    ax.text((x_fs + x_cnn) / 2, y_top + 0.003, sig,
                            ha='center', va='bottom', fontsize=12,
                            fontweight='bold')

                ax.axhline(0.5, color='grey', linestyle='--', lw=1.2, alpha=0.7,
                           label='Chance (0.5)')
                ax.set_xticks(x)
                ax.set_xticklabels(BANDS_SPAN_C, fontsize=11)
                ax.set(ylabel=metric_label if col == 0 else '',
                       title=CLF_LABELS_C[clf], ylim=(0.3, 1.05))
                if row == 0 and col == 0:
                    ax.legend(fontsize=9, loc='upper right')

            plt.setp(axes[row][1].get_yticklabels(), visible=False)

        for row, (_, metric_label) in enumerate(metrics_c):
            axes[row][0].set_ylabel(metric_label, fontsize=11)

        plt.suptitle(
            'G1 ([f_first, f_mid, f_last]) vs G2 ([f_first, f_mid, f_mid]) — Concatenation Span\n'
            'mean ± std over 5 folds  (* p<0.05, Wilcoxon, CNN vs FS ROI)',
            fontsize=12, fontweight='bold'
        )
        plt.tight_layout()
        plt.savefig(root_dir / 'brainage_agg/outputs/figures/span_concat_summary.png',
                    dpi=150, bbox_inches='tight')
        plt.show()


### 16e  Difference vs Concatenation — direct variant comparison

Both variants test whether full-span features are distinguishable from short-span features.

| Variant | G1 (full span) | G2 (short span) | Feature dim |
|---|---|---|---|
| **Difference** | `f_last − f_first` | `f_mid − f_first` or `f_last − f_mid` | n |
| **Concatenation** | `[f_first, f_mid, f_last]` | `[f_first, f_mid, f_mid]` or `[f_mid, f_mid, f_last]` | 3n |

Grouped bars per band: Difference (blue tones) vs Concatenation (green tones); FS ROI solid, CNN hatched.
Stars = one-sample t-test vs chance=0.5, BH-FDR corrected.

In [ ]:
# ── §16e  Difference vs Concatenation — direct variant comparison ─────────────
_span_path = root_dir / 'brainage_agg/outputs/span_classif_results.csv'

if not _span_path.exists():
    print('span_classif_results.csv not found.')
elif 'variant' not in pd.read_csv(_span_path, nrows=1).columns:
    print('No variant column — re-run SLURM jobs.')
else:
    from statsmodels.stats.multitest import multipletests as _mht

    _sp_e = pd.read_csv(_span_path)
    if 'variant' not in _sp_e.columns:
        _sp_e['variant'] = 'difference'

    _VARIANTS_E  = ['difference', 'concatenation']
    _VAR_COLORS  = {
        'difference':    {'fs_roi': '#1565C0', 'cnn': '#0D47A1'},
        'concatenation': {'fs_roi': '#2E7D32', 'cnn': '#1B5E20'},
    }
    _VAR_ALPHA  = {'fs_roi': 0.80, 'cnn': 0.55}
    _VAR_HATCH  = {'fs_roi': '',   'cnn': '///'}
    _EXT_ORDER  = ['fs_roi', 'cnn']
    _EXT_LBL    = {'fs_roi': 'FS ROI', 'cnn': 'CNN (best)'}
    _BANDS_E    = ['Child', 'Adult']
    _CLFS_E     = ['LogisticReg', 'RandomForest']
    _CLF_LBL_E  = {'LogisticReg': 'Logistic Regression', 'RandomForest': 'Random Forest'}
    _METRICS_E  = [('bacc', 'Balanced Accuracy'), ('auc', 'AUC-ROC')]

    def _folds_e(var, ext, band, clf, metric):
        sub = _sp_e[(_sp_e['variant'] == var) &
                    (_sp_e['extractor_type'] == ext) &
                    (_sp_e['band'] == band) &
                    (_sp_e['classifier'] == clf)]
        if ext == 'fs_roi':
            return sub.groupby('fold')[metric].mean()
        seed_avg = (sub.groupby(['cnn_arch', 'scaler', 'channels', 'fold'])[metric]
                    .mean().reset_index())
        cfg_mean = seed_avg.groupby(['cnn_arch', 'scaler', 'channels'])[metric].mean()
        b = cfg_mean.idxmax()
        return (seed_avg[(seed_avg['cnn_arch'] == b[0]) &
                         (seed_avg['scaler']   == b[1]) &
                         (seed_avg['channels'] == b[2])]
                .set_index('fold')[metric])

    # bar order: 4 bars per band-group: diff×FS, diff×CNN, concat×FS, concat×CNN
    _bar_specs = [(v, e) for v in _VARIANTS_E for e in _EXT_ORDER]
    _n_bars = len(_bar_specs)   # 4
    _w = 0.18
    # offsets: -1.5, -0.5 (diff group), +0.5, +1.5 (concat group) × w
    _offsets = np.array([-1.5, -0.5, 0.5, 1.5]) * _w

    for metric, metric_lbl in _METRICS_E:
        fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

        # collect all raw p-values for BH-FDR per metric across both panels
        _all_pv = []   # (ax, xi, bar_i, mean_val, std_val, p_raw)

        for col, clf in enumerate(_CLFS_E):
            ax = axes[col]
            _x = np.arange(len(_BANDS_E))

            for i, (var, ext) in enumerate(_bar_specs):
                means, stds = [], []
                for band in _BANDS_E:
                    fv = _folds_e(var, ext, band, clf, metric)
                    means.append(float(fv.mean()) if len(fv) else np.nan)
                    stds.append(float(fv.std())  if len(fv) else 0.0)

                color  = _VAR_COLORS[var][ext]
                hatch  = _VAR_HATCH[ext]
                alpha  = _VAR_ALPHA[ext]
                ax.bar(_x + _offsets[i], means, _w,
                       color=color, hatch=hatch, alpha=alpha,
                       yerr=stds, capsize=3,
                       error_kw={'lw': 1.0, 'capthick': 1.0},
                       edgecolor=color, linewidth=0.7, zorder=3)

                for xi, band in enumerate(_BANDS_E):
                    fv = _folds_e(var, ext, band, clf, metric)
                    if len(fv) >= 3:
                        p = scipy_stats.ttest_1samp(
                            fv.values, popmean=0.5, alternative='greater').pvalue
                        _all_pv.append((ax, xi, i, means[xi], stds[xi], p))

            # dashed separator between variant groups within each band
            for xi in range(len(_BANDS_E)):
                ax.axvline(xi, color='#BDBDBD', lw=0.8, ls=':', zorder=1)

            ax.axhline(0.5, color='grey', ls='--', lw=1.2, alpha=0.6, zorder=2)
            ax.set_xticks(_x)
            ax.set_xticklabels(_BANDS_E, fontsize=11)
            if col == 0:
                ax.set_ylabel(metric_lbl, fontsize=11)
            ax.set_title(_CLF_LBL_E[clf], fontsize=12, fontweight='bold')
            ax.set_ylim(0.35, 1.02)
            ax.grid(axis='y', alpha=0.25, zorder=0)

            # sub-group labels below x-axis (text only, no bracket lines)
            for gi, (var, _vlbl) in enumerate([('difference', 'Difference'),
                                               ('concatenation', 'Concatenation')]):
                xc = (_offsets[gi * 2] + _offsets[gi * 2 + 1]) / 2
                ax.text(np.mean(_x) + xc, -0.09,
                        _vlbl,
                        ha='center', va='top', fontsize=8.5,
                        transform=ax.get_xaxis_transform(),
                        color='#424242')

        # BH-FDR correction over all collected p-values for this metric
        if _all_pv:
            _praw = np.array([t[5] for t in _all_pv])
            _rej, _, _, _ = _mht(_praw, alpha=0.05, method='fdr_bh')
            for (t_ax, t_xi, t_i, t_m, t_s, _), t_sig in zip(_all_pv, _rej):
                if t_sig and not np.isnan(t_m):
                    t_ax.text(t_xi + _offsets[t_i], t_m + t_s + 0.008,
                              f'{t_m:.2f}', ha='center', va='bottom',
                              fontsize=9, fontweight='bold',
                              color=_VAR_COLORS[_bar_specs[t_i][0]][_bar_specs[t_i][1]])
                    t_ax.text(t_xi + _offsets[t_i], t_m + t_s + 0.028,
                              '*', ha='center', va='bottom',
                              fontsize=9, fontweight='bold',
                              color=_VAR_COLORS[_bar_specs[t_i][0]][_bar_specs[t_i][1]])

        # legend on first panel
        import matplotlib.patches as _mpa
        _handles = []
        for var, vlbl in [('difference', 'Difference'), ('concatenation', 'Concatenation')]:
            for ext, elbl in [('fs_roi', 'FS ROI'), ('cnn', 'CNN best')]:
                _handles.append(_mpa.Patch(
                    facecolor=_VAR_COLORS[var][ext],
                    hatch=_VAR_HATCH[ext],
                    alpha=_VAR_ALPHA[ext],
                    label=f'{vlbl} / {elbl}'))
        _handles.append(plt.Line2D([0],[0], color='grey', ls='--', lw=1.2,
                                   label='Chance (0.5)'))
        axes[0].legend(handles=_handles, fontsize=8, loc='upper right', ncol=2)

        plt.suptitle(
            f'Span Classification — Difference vs Concatenation  ({metric_lbl})\n'
            '* = one-sample t-test vs chance=0.5, BH-FDR q<0.05  |  mean ± std over 5 folds',
            fontsize=12, fontweight='bold')
        plt.tight_layout()
        fig.savefig(root_dir / f'brainage_agg/outputs/figures/span_variant_comparison_{metric}.png',
                    dpi=150, bbox_inches='tight')
        plt.show()


---
## 17  η² (effect size) — linear relation between aggregated features and age

For each aggregation, we aggregate the FS ROI features for all eligible subjects
(no CV split — diagnostic only) and compute the **η²** (omega-squared ≈ R²) from
`feature_k ~ age` OLS for every feature dimension k.

η² is sample-size-corrected (unlike raw F-values, which inflate with N), allowing
fair comparison across aggregations that have different eligible cohort sizes
(`concatenation` requires ≥3 tp, so n ≈ 148 vs n ≈ 232 for `mean`).

High η² means the aggregation preserves information that is linearly decodable from
chronological age — necessary but not sufficient for good Ridge regression performance.


In [ ]:
import sys; sys.path.insert(0, str(root_dir))
from sklearn.feature_selection import f_regression
from brainage_agg.data.manifest import build_manifest
from brainage_agg.features.loader import load_features, align_to_manifest
from brainage_agg.agg import aggregations as agg_mod

_fs_npz = root_dir / 'outputs/features/features__model-freesurfer_roi__scaler-none__channels-all_roi__seed-0.npz'
_manifest_f = build_manifest(_fs_npz, t_mid_seed=42)
X_fs_raw, meta_fs, _ = load_features(_fs_npz)
subj_data_fs = align_to_manifest(X_fs_raw, meta_fs, _manifest_f)

_res_manifest = pd.read_csv(root_dir / 'brainage_agg/outputs/manifest.csv')
_manifest_f = _manifest_f.merge(
    _res_manifest[['subject_id','brain_change_rate']].drop_duplicates(),
    on='subject_id', how='left'
)

from sklearn.preprocessing import StandardScaler as _SS

def _compute_eta2(X, y):
    """η² per feature from f_regression against target y."""
    F, _ = f_regression(X, y)
    return F / (F + len(y) - 2)

def _agg_subjects(manifest_sub, subj_data, aggregation, target):
    """Aggregate features per subject. target: 'age', 'brain_change_rate', or 'band'."""
    spec = agg_mod.AGGREGATION_SPECS[aggregation]
    cohort_col = 'cohort_concat' if spec['cohort'] == 'concat' else 'cohort_all'
    mdf = manifest_sub[manifest_sub[cohort_col] & manifest_sub['band'].isin(['Child','Adult'])].copy()
    if target == 'brain_change_rate' or aggregation in ('annualized_rate','difference','lme_slope_change'):
        mdf = mdf[~mdf['exclude_arm_b']]
    if target == 'brain_change_rate':
        mdf = mdf[mdf['brain_change_rate'].notna()]

    all_flat = np.vstack([subj_data[s]['features'] for s in mdf['subject_id'] if s in subj_data])
    sc = _SS().fit(all_flat)

    X_rows, y_rows = [], []
    for _, row in mdf.iterrows():
        sid = row['subject_id']
        if sid not in subj_data: continue
        feats = sc.transform(subj_data[sid]['features'])
        ages  = subj_data[sid]['ages']
        if   aggregation == 'mean':          x = agg_mod.mean(feats)
        elif aggregation == 'difference':    x = agg_mod.difference(feats)
        elif aggregation == 'annualized_rate':
            dt = float(ages[-1] - ages[0])
            if abs(dt) < 1e-8: continue
            x = agg_mod.annualized_rate(feats, ages)
        elif aggregation == 'concatenation':
            gri = np.asarray(row['row_indices'] if isinstance(row['row_indices'], list) else list(row['row_indices']))
            loc = np.where(gri == int(row['t_mid_idx']))[0]
            if len(loc) == 0: continue
            x = agg_mod.concatenation(feats, 0, int(loc[0]), len(feats)-1)
        else: continue
        X_rows.append(x)
        if   target == 'age':               y_rows.append(float(np.mean(ages)))
        elif target == 'brain_change_rate': y_rows.append(float(row['brain_change_rate']))
        elif target == 'band':              y_rows.append(0.0 if row['band'] == 'Child' else 1.0)
    return np.array(X_rows, dtype=np.float32), np.array(y_rows)

def _span_g1g2(manifest_sub, subj_data):
    """G1 = f_last - f_first (full span), G2 = f_mid - f_first (half span).
    Returns stacked (X, y) where y=1 for G1, y=0 for G2.
    Requires cohort_concat (n_tp >= 3).
    """
    mdf = manifest_sub[manifest_sub['cohort_concat'] & manifest_sub['band'].isin(['Child','Adult'])].copy()
    all_flat = np.vstack([subj_data[s]['features'] for s in mdf['subject_id'] if s in subj_data])
    sc = _SS().fit(all_flat)
    X_rows, y_rows = [], []
    for _, row in mdf.iterrows():
        sid = row['subject_id']
        if sid not in subj_data: continue
        feats = sc.transform(subj_data[sid]['features'])  # (n_tp, n_feat)
        gri = np.asarray(row['row_indices'] if isinstance(row['row_indices'], list) else list(row['row_indices']))
        loc = np.where(gri == int(row['t_mid_idx']))[0]
        if len(loc) == 0: continue
        t_mid = int(loc[0])
        g1 = feats[-1] - feats[0]   # full span difference
        g2 = feats[t_mid] - feats[0]  # half span difference
        X_rows.append(g1); y_rows.append(1.0)
        X_rows.append(g2); y_rows.append(0.0)
    return np.array(X_rows, dtype=np.float32), np.array(y_rows)

AGG_ORDER = ['mean', 'concatenation', 'annualized_rate', 'difference']

# Compute η² for all four panels
panels = {
    'arm_a': {}, 'arm_b': {}, 'child_adult': {}
}
for agg in AGG_ORDER:
    for key, tgt in [('arm_a','age'), ('arm_b','brain_change_rate'), ('child_adult','band')]:
        X, y = _agg_subjects(_manifest_f, subj_data_fs, agg, tgt)
        if len(X) >= 10:
            panels[key][agg] = _compute_eta2(X, y)

X_span, y_span = _span_g1g2(_manifest_f, subj_data_fs)
eta2_span = _compute_eta2(X_span, y_span) if len(X_span) >= 10 else None

print('η² medians by panel and aggregation:')
print(f'  {"":22s}  {"Arm A":>8}  {"Arm B":>8}  {"Child/Adult":>11}')
for agg in AGG_ORDER:
    a = f"{np.median(panels['arm_a'].get(agg, [np.nan])):.4f}"
    b = f"{np.median(panels['arm_b'].get(agg, [np.nan])):.4f}"
    c = f"{np.median(panels['child_adult'].get(agg, [np.nan])):.4f}"
    print(f'  {agg:22s}  {a:>8}  {b:>8}  {c:>11}')
if eta2_span is not None:
    print(f'  Span G1 vs G2 (n={len(X_span)}): median η²={np.median(eta2_span):.4f}')

# ── 2×2 figure ──────────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 9), sharey=True)

PANEL_SPECS = [
    (axes[0,0], panels['arm_a'],     AGG_ORDER, 'Arm A — target: age',
     ['mean','concatenation'],       'Expected winner: state aggs'),
    (axes[0,1], panels['arm_b'],     AGG_ORDER, 'Arm B — target: brain change rate',
     ['annualized_rate','difference'], 'Expected winner: change aggs'),
    (axes[1,0], panels['child_adult'], AGG_ORDER, 'Child/Adult — target: band label (0/1)',
     ['mean','concatenation'],       'Expected winner: state aggs'),
    (axes[1,1], None,                None,      'Span — G1 (full) vs G2 (half) difference',
     None,                           'Single analysis (no aggregation axis)'),
]

for ax, eta2_dict, agg_order, title, winners, note in PANEL_SPECS:
    if eta2_dict is not None:
        avail = [a for a in agg_order if a in eta2_dict]
        colors = ['#43A047' if a in (winners or []) else '#E53935' for a in avail]
        parts = ax.violinplot(
            [eta2_dict[a] for a in avail],
            positions=range(len(avail)),
            showmedians=True, showextrema=False,
        )
        for pc, c in zip(parts['bodies'], colors):
            pc.set_facecolor(c); pc.set_alpha(0.55)
        parts['cmedians'].set_color('black'); parts['cmedians'].set_lw(2)
        ax.set_xticks(range(len(avail)))
        ax.set_xticklabels([AGG_LABELS.get(a, a) for a in avail], fontsize=10)
    else:
        # Span panel: single violin
        if eta2_span is not None:
            parts = ax.violinplot([eta2_span], positions=[0],
                                  showmedians=True, showextrema=False)
            parts['bodies'][0].set_facecolor('#7B1FA2'); parts['bodies'][0].set_alpha(0.55)
            parts['cmedians'].set_color('black'); parts['cmedians'].set_lw(2)
            ax.set_xticks([0])
            ax.set_xticklabels([f'G1 vs G2\n(n={len(X_span)} diff vectors)'], fontsize=10)
            ax.text(0, np.median(eta2_span) + 0.02,
                    f'median={np.median(eta2_span):.3f}',
                    ha='center', va='bottom', fontsize=9, fontweight='bold')
        else:
            ax.text(0.5, 0.5, 'Insufficient data', ha='center', va='center',
                    transform=ax.transAxes, fontsize=11)

    ax.set(ylabel='η² per feature', ylim=(-0.02, 1.0), title=title)
    ax.axhline(0.05, color='grey', linestyle=':', lw=1.2)
    ax.text(0.98, 0.97, note, transform=ax.transAxes,
            ha='right', va='top', fontsize=8, color='#555',
            bbox=dict(boxstyle='round,pad=0.2', fc='white', ec='#ccc', alpha=0.8))

axes[0,0].legend(handles=[
    mpatches.Patch(color='#43A047', label='Expected winner'),
    mpatches.Patch(color='#E53935', label='Expected loser'),
    mpatches.Patch(color='#7B1FA2', label='Span G1/G2'),
], fontsize=9, loc='upper right')

plt.suptitle(
    'FS ROI — η² (feature–target linear association) by aggregation and task\n'
    'Each violin = distribution of η² across all feature dimensions',
    fontsize=12, fontweight='bold'
)
plt.tight_layout()
plt.savefig(root_dir / 'brainage_agg/outputs/figures/eta2_by_aggregation.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ── Top-k features by η² for each panel ─────────────────────────────────────
import json as _json
_meta = _json.loads((root_dir / 'outputs/features/features__model-freesurfer_roi__scaler-none__channels-all_roi__seed-0.json').read_text())
feat_names = np.array(_meta['feature_names'])

K = 15  # top features to show per panel

def _best_agg(eta2_dict, candidates):
    avail = {a: np.median(v) for a, v in eta2_dict.items() if a in candidates}
    return max(avail, key=avail.get) if avail else None

best_a  = _best_agg(panels['arm_a'],       ['mean', 'concatenation'])
best_b  = _best_agg(panels['arm_b'],       ['annualized_rate', 'difference'])
best_ca = _best_agg(panels['child_adult'], ['mean', 'concatenation'])

panel_defs = []
if best_a  and best_a  in panels['arm_a']:       panel_defs.append(('Arm A',         panels['arm_a'][best_a],       best_a,  '#43A047'))
if best_b  and best_b  in panels['arm_b']:       panel_defs.append(('Arm B',         panels['arm_b'][best_b],       best_b,  '#1565C0'))
if best_ca and best_ca in panels['child_adult']: panel_defs.append(('Child/Adult',   panels['child_adult'][best_ca], best_ca, '#E65100'))
if eta2_span is not None:                         panel_defs.append(('Span G1 vs G2', eta2_span,                    'difference', '#7B1FA2'))

fig, axes = plt.subplots(2, 2, figsize=(14, 12))
axes_flat = axes.flat

for ax, (panel_name, eta2_vals, agg_label, color) in zip(axes_flat, panel_defs):
    if agg_label == 'concatenation' and len(eta2_vals) == 3 * len(feat_names):
        names_arr = np.array(
            [f + ' [t_first]' for f in feat_names] +
            [f + ' [t_mid]'   for f in feat_names] +
            [f + ' [t_last]'  for f in feat_names]
        )
    else:
        names_arr = feat_names

    n = min(len(eta2_vals), len(names_arr))
    top_idx   = np.argsort(eta2_vals[:n])[-K:][::-1]
    top_eta2  = eta2_vals[top_idx]
    top_names = names_arr[top_idx]

    short = [n.replace('Left-','L-').replace('Right-','R-')
               .replace('_ThickAvg','_Thk').replace('_GrayVol','_GV')
               .replace('_SurfArea','_SA').replace('_MeanCurv','_MC')
               .replace('lh_','lh/').replace('rh_','rh/')
             for n in top_names]

    y_pos = np.arange(K)
    ax.barh(y_pos, top_eta2, color=color, alpha=0.75, edgecolor='white')
    ax.set_yticks(y_pos)
    ax.set_yticklabels(short, fontsize=8)
    ax.invert_yaxis()
    ax.axvline(0.05, color='grey', linestyle=':', lw=1.2)
    ax.set(xlabel='η²', title=f'{panel_name}  —  agg: {AGG_LABELS.get(agg_label, agg_label)}')
    for i, v in enumerate(top_eta2):
        ax.text(v + 0.003, i, f'{v:.3f}', va='center', fontsize=7.5)

# Hide unused axes if fewer than 4 panels
for ax in list(axes_flat)[len(panel_defs):]:
    ax.set_visible(False)

plt.suptitle(f'Top-{K} features by η²  —  FS ROI  (best expected-winner aggregation per panel)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(root_dir / 'brainage_agg/outputs/figures/eta2_top_features.png', dpi=150, bbox_inches='tight')
plt.show()

# ── Printed table ────────────────────────────────────────────────────────────
for panel_name, eta2_vals, agg_label, _ in panel_defs:
    n = min(len(eta2_vals), len(feat_names))
    top_idx = np.argsort(eta2_vals[:n])[-10:][::-1]
    print(f'\n{panel_name} ({AGG_LABELS.get(agg_label, agg_label)}) — top 10:')
    for rank, i in enumerate(top_idx, 1):
        print(f'  {rank:2d}. {feat_names[i]:45s}  η²={eta2_vals[i]:.4f}')


### 17b  η² — CNN features (best config)

Same analysis repeated for the best CNN extractor (lowest mean MAE across all arms).  
CNN feature dimensions have no semantic labels — top-k shows dimension indices.  
Compare median η² against FS ROI to assess whether CNN features encode more or less target-relevant signal.


In [ ]:
from brainage_agg.features.loader import load_features, align_to_manifest

# ── Load best CNN extractor ───────────────────────────────────────────────────
_feat_dir = root_dir / 'outputs/features'
_cnn_rank = (
    df[(df['extractor_type']=='cnn') & (df['age_band']=='all')]
    .groupby('extractor')['MAE'].mean()
)
_best_cnn_name = _cnn_rank.idxmin()
_best_cnn_row  = df[df['extractor']==_best_cnn_name].iloc[0]
print(f'Best CNN: {_best_cnn_name}')
print(f'  arch={_best_cnn_row["cnn_arch"]}  scaler={_best_cnn_row["scaler"]}  channels={_best_cnn_row["channels"]}')

_cnn_npz = _feat_dir / f'{_best_cnn_name}.npz'
_manifest_cnn = build_manifest(_cnn_npz, t_mid_seed=42)
X_cnn_raw, meta_cnn, _ = load_features(_cnn_npz)
subj_data_cnn = align_to_manifest(X_cnn_raw, meta_cnn, _manifest_cnn)

_manifest_cnn = _manifest_cnn.merge(
    _res_manifest[['subject_id','brain_change_rate']].drop_duplicates(),
    on='subject_id', how='left'
)
n_cnn_feats = X_cnn_raw.shape[1]
print(f'CNN feature dims: {n_cnn_feats}')

# ── Compute η² for CNN ───────────────────────────────────────────────────────
panels_cnn = {'arm_a': {}, 'arm_b': {}, 'child_adult': {}}
for agg in AGG_ORDER:
    for key, tgt in [('arm_a','age'), ('arm_b','brain_change_rate'), ('child_adult','band')]:
        X, y = _agg_subjects(_manifest_cnn, subj_data_cnn, agg, tgt)
        if len(X) >= 10:
            panels_cnn[key][agg] = _compute_eta2(X, y)

X_span_cnn, y_span_cnn = _span_g1g2(_manifest_cnn, subj_data_cnn)
eta2_span_cnn = _compute_eta2(X_span_cnn, y_span_cnn) if len(X_span_cnn) >= 10 else None

# ── 2×2 violin: CNN ─────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 9), sharey=True)

PANEL_SPECS_CNN = [
    (axes[0,0], panels_cnn['arm_a'],      AGG_ORDER, 'Arm A — target: age',
     ['mean','concatenation'],             'Expected winner: state aggs'),
    (axes[0,1], panels_cnn['arm_b'],      AGG_ORDER, 'Arm B — target: brain change rate',
     ['annualized_rate','difference'],     'Expected winner: change aggs'),
    (axes[1,0], panels_cnn['child_adult'], AGG_ORDER, 'Child/Adult — target: band label',
     ['mean','concatenation'],             'Expected winner: state aggs'),
    (axes[1,1], None,                     None,      'Span — G1 vs G2 (CNN)',
     None,                                'Single analysis (no aggregation axis)'),
]

for ax, eta2_dict, agg_order, title, winners, note in PANEL_SPECS_CNN:
    if eta2_dict is not None:
        avail = [a for a in agg_order if a in eta2_dict]
        colors = ['#43A047' if a in (winners or []) else '#E53935' for a in avail]
        parts = ax.violinplot(
            [eta2_dict[a] for a in avail],
            positions=range(len(avail)),
            showmedians=True, showextrema=False,
        )
        for pc, c in zip(parts['bodies'], colors):
            pc.set_facecolor(c); pc.set_alpha(0.55)
        parts['cmedians'].set_color('black'); parts['cmedians'].set_lw(2)
        ax.set_xticks(range(len(avail)))
        ax.set_xticklabels([AGG_LABELS.get(a, a) for a in avail], fontsize=10)
    else:
        if eta2_span_cnn is not None:
            parts = ax.violinplot([eta2_span_cnn], positions=[0],
                                  showmedians=True, showextrema=False)
            parts['bodies'][0].set_facecolor('#7B1FA2'); parts['bodies'][0].set_alpha(0.55)
            parts['cmedians'].set_color('black'); parts['cmedians'].set_lw(2)
            ax.set_xticks([0])
            ax.set_xticklabels([f'G1 vs G2\n(n={len(X_span_cnn)})'], fontsize=10)
            ax.text(0, np.median(eta2_span_cnn)+0.02,
                    f'median={np.median(eta2_span_cnn):.3f}',
                    ha='center', va='bottom', fontsize=9, fontweight='bold')
    ax.set(ylabel='η² per feature', ylim=(-0.02, 1.0), title=title)
    ax.axhline(0.05, color='grey', linestyle=':', lw=1.2)
    ax.text(0.98, 0.97, note, transform=ax.transAxes,
            ha='right', va='top', fontsize=8, color='#555',
            bbox=dict(boxstyle='round,pad=0.2', fc='white', ec='#ccc', alpha=0.8))

axes[0,0].legend(handles=[
    mpatches.Patch(color='#43A047', label='Expected winner'),
    mpatches.Patch(color='#E53935', label='Expected loser'),
    mpatches.Patch(color='#7B1FA2', label='Span G1/G2'),
], fontsize=9)
plt.suptitle(
    f'CNN ({_best_cnn_row["cnn_arch"]} / {_best_cnn_row["scaler"]} / {_best_cnn_row["channels"]}) — '
    f'η² by aggregation and task\n'
    f'{n_cnn_feats} feature dims  (vs {len(feat_names)} FS ROI dims)',
    fontsize=12, fontweight='bold'
)
plt.tight_layout()
plt.savefig(root_dir / 'brainage_agg/outputs/figures/eta2_cnn_by_aggregation.png', dpi=150, bbox_inches='tight')
plt.show()

# ── 2×2 top-k CNN dimensions ─────────────────────────────────────────────────
best_a_cnn  = _best_agg(panels_cnn['arm_a'],       ['mean','concatenation'])
best_b_cnn  = _best_agg(panels_cnn['arm_b'],       ['annualized_rate','difference'])
best_ca_cnn = _best_agg(panels_cnn['child_adult'], ['mean','concatenation'])

panel_defs_cnn = []
if best_a_cnn  and best_a_cnn  in panels_cnn['arm_a']:       panel_defs_cnn.append(('Arm A',         panels_cnn['arm_a'][best_a_cnn],       best_a_cnn,  '#43A047'))
if best_b_cnn  and best_b_cnn  in panels_cnn['arm_b']:       panel_defs_cnn.append(('Arm B',         panels_cnn['arm_b'][best_b_cnn],       best_b_cnn,  '#1565C0'))
if best_ca_cnn and best_ca_cnn in panels_cnn['child_adult']: panel_defs_cnn.append(('Child/Adult',   panels_cnn['child_adult'][best_ca_cnn], best_ca_cnn, '#E65100'))
if eta2_span_cnn is not None:                                  panel_defs_cnn.append(('Span G1 vs G2', eta2_span_cnn,                        'difference', '#7B1FA2'))

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for ax, (panel_name, eta2_vals, agg_label, color) in zip(axes.flat, panel_defs_cnn):
    n = len(eta2_vals)
    top_idx  = np.argsort(eta2_vals)[-K:][::-1]
    top_eta2 = eta2_vals[top_idx]
    labels   = [f'dim {i}' for i in top_idx]
    y_pos = np.arange(K)
    ax.barh(y_pos, top_eta2, color=color, alpha=0.75, edgecolor='white')
    ax.set_yticks(y_pos)
    ax.set_yticklabels(labels, fontsize=8)
    ax.invert_yaxis()
    ax.axvline(0.05, color='grey', linestyle=':', lw=1.2)
    ax.set(xlabel='η²', title=f'{panel_name}  —  agg: {AGG_LABELS.get(agg_label, agg_label)}')
    for i, v in enumerate(top_eta2):
        ax.text(v + 0.001, i, f'{v:.3f}', va='center', fontsize=7.5)

for ax in list(axes.flat)[len(panel_defs_cnn):]:
    ax.set_visible(False)

plt.suptitle(
    f'Top-{K} CNN dimensions by η²  —  {_best_cnn_row["cnn_arch"]} / {_best_cnn_row["scaler"]} / {_best_cnn_row["channels"]}',
    fontsize=12, fontweight='bold'
)
plt.tight_layout()
plt.savefig(root_dir / 'brainage_agg/outputs/figures/eta2_cnn_top_dims.png', dpi=150, bbox_inches='tight')
plt.show()

# ── FS ROI vs CNN median η² comparison ───────────────────────────────────────
print('\nMedian η² comparison  (FS ROI vs CNN):')
print(f'  {"":22s}  {"Arm A":>14}  {"Arm B":>14}  {"Child/Adult":>14}')
print(f'  {"":22s}  {"ROI":>6} {"CNN":>6}  {"ROI":>6} {"CNN":>6}  {"ROI":>6} {"CNN":>6}')
for agg in AGG_ORDER:
    row_parts = [f'{agg:22s}']
    for key in ['arm_a', 'arm_b', 'child_adult']:
        roi_med = f'{np.median(panels[key][agg]):.4f}'     if agg in panels[key]     else '  —   '
        cnn_med = f'{np.median(panels_cnn[key][agg]):.4f}' if agg in panels_cnn[key] else '  —   '
        row_parts.append(f'{roi_med:>6} {cnn_med:>6}')
    print('  ' + '  '.join(row_parts))


---
## 18  Numerical stability — catastrophic cancellation: CNN vs FS ROI, across aggregations

For difference-based aggregations, the feature vector `f_last − f_first` loses precision
when the change is small relative to the absolute feature magnitude.

**Cancellation score** per subject:
```
cancellation_score = median_k( |f_last_k − f_first_k| / max(|f_last_k|, |f_first_k|, ε) )
```
Near 0 → high cancellation risk (large values, tiny change). Near 1 → no cancellation.

We compare **FS ROI** (raw mm³) against the **best CNN per scaler** (lowest mean MAE
on Arm B difference), and show whether the pattern holds across all Arm B aggregations.
State aggregations (mean, concatenation) are immune to cancellation — they appear as
a baseline.


In [ ]:
_feat_dir = root_dir / 'outputs/features'
_fs_npz   = _feat_dir / 'features__model-freesurfer_roi__scaler-none__channels-all_roi__seed-0.npz'

def _cancellation_score(npz_path, manifest_sub):
    """Mean (across subjects) of the per-subject median relative feature change."""
    from brainage_agg.features.loader import load_features, align_to_manifest
    X_raw, meta_df, _ = load_features(npz_path)
    subj_data = align_to_manifest(X_raw, meta_df, manifest_sub)
    mdf = manifest_sub[manifest_sub['cohort_all'] & ~manifest_sub['exclude_arm_b']].copy()
    scores = []
    for _, row in mdf.iterrows():
        sid = row['subject_id']
        if sid not in subj_data: continue
        feats  = subj_data[sid]['features'].astype(np.float64)
        diff   = np.abs(feats[-1] - feats[0])
        denom  = np.maximum(np.maximum(np.abs(feats[-1]), np.abs(feats[0])), 1e-6)
        scores.append(float(np.median(diff / denom)))
    return float(np.mean(scores)) if scores else np.nan

# ── FS ROI ────────────────────────────────────────────────────────────────────
fs_canc = _cancellation_score(_fs_npz, _manifest_f)
print(f"FS ROI  cancellation={fs_canc:.4f}")

# ── Best CNN per scaler (min mean MAE on Arm B difference, all band) ──────────
_arm_b = (
    df[(df['target_arm']=='B') & (df['aggregation']=='difference') &
       (df['age_band']=='all') & (df['extractor_type']=='cnn')]
    .groupby(['extractor','scaler'])['MAE'].mean().reset_index()
)
best_per_scaler = _arm_b.loc[_arm_b.groupby('scaler')['MAE'].idxmin()].reset_index(drop=True)

model_rows = [{'model': 'FS ROI', 'scaler': 'none', 'cancellation': fs_canc,
               'extractor': _fs_npz.stem, 'extractor_type': 'fs_roi'}]
for _, r in best_per_scaler.iterrows():
    npz = _feat_dir / f"{r['extractor']}.npz"
    if not npz.exists(): continue
    canc = _cancellation_score(npz, _manifest_f)
    print(f"CNN best {r['scaler']:8s}  cancellation={canc:.4f}  ({r['extractor']})")
    model_rows.append({'model': f"CNN\n({r['scaler']})", 'scaler': r['scaler'],
                       'cancellation': canc, 'extractor': r['extractor'],
                       'extractor_type': 'cnn'})

model_df = pd.DataFrame(model_rows)

# ── MAE from results.csv for all Arm B aggregations ───────────────────────────
AGG_ORDER_NS = ['annualized_rate', 'difference', 'lme_slope_change', 'mean', 'concatenation']
AGG_CHANGE   = {'annualized_rate', 'difference', 'lme_slope_change'}

mae_rows = []
for _, m in model_df.iterrows():
    for agg in AGG_ORDER_NS:
        sub = df[(df['target_arm']=='B') & (df['aggregation']==agg) &
                 (df['age_band']=='all') & (df['extractor']==m['extractor'])]
        if len(sub) == 0: continue
        mae_rows.append({'model': m['model'], 'scaler': m['scaler'],
                         'aggregation': agg, 'MAE': sub['MAE'].mean()})

mae_df = pd.DataFrame(mae_rows)

# ── Figure ────────────────────────────────────────────────────────────────────
_COLORS = {'none':   '#2196F3',
           'minmax': SCALER_COLORS['minmax'],
           'zscore': SCALER_COLORS['zscore'],
           'robust': SCALER_COLORS['robust']}

fig = plt.figure(figsize=(16, 6))
gs  = fig.add_gridspec(1, 6, wspace=0.45)
ax_canc = fig.add_subplot(gs[0, :2])
ax_mae  = fig.add_subplot(gs[0, 2:])

# ── Panel 1: cancellation scores ─────────────────────────────────────────────
x   = np.arange(len(model_df))
bar_colors = [_COLORS[r['scaler']] for _, r in model_df.iterrows()]
bars = ax_canc.bar(x, model_df['cancellation'], color=bar_colors, alpha=0.85,
                   width=0.55, edgecolor='white')
ax_canc.set_xticks(x)
ax_canc.set_xticklabels(model_df['model'], fontsize=9)
ax_canc.set(ylabel='Cancellation score (higher = more stable)',
            title='Numerical stability\n(property of features, not aggregation)')
for bar, v in zip(bars, model_df['cancellation']):
    ax_canc.text(bar.get_x() + bar.get_width()/2,
                 bar.get_height() + 0.002,
                 f'{v:.3f}', ha='center', va='bottom', fontsize=8, fontweight='bold')

# ── Panel 2: MAE heatmap (aggregation × model) ────────────────────────────────
pivot = mae_df.pivot(index='aggregation', columns='model', values='MAE')
pivot = pivot.reindex([a for a in AGG_ORDER_NS if a in pivot.index])
pivot.index = [AGG_LABELS.get(a, a) for a in pivot.index]

_valid = pivot.notna()
_vmin  = pivot.values[np.isfinite(pivot.values)].min() * 0.97
_vmax  = pivot.values[np.isfinite(pivot.values)].max() * 1.03
sns.heatmap(pivot, annot=True, fmt='.3f', cmap='RdYlGn_r', ax=ax_mae,
            mask=~_valid, vmin=_vmin, vmax=_vmax,
            linewidths=0.5, linecolor='white',
            cbar_kws={'label': 'MAE (Arm B)', 'shrink': 0.8},
            annot_kws={'size': 9, 'weight': 'bold'})
ax_mae.set_title('MAE by aggregation × model\n'
                 '(green rows = state aggs, immune to cancellation)',
                 fontsize=10, fontweight='bold')
ax_mae.set(xlabel='', ylabel='')
ax_mae.tick_params(axis='both', rotation=0)

# Shade state aggregation rows
for i, agg in enumerate(pivot.index):
    raw_agg = [a for a in AGG_ORDER_NS if AGG_LABELS.get(a,a) == agg]
    if raw_agg and raw_agg[0] not in AGG_CHANGE:
        ax_mae.axhspan(i, i+1, color='#E8F5E9', alpha=0.4, zorder=0)

plt.suptitle('Numerical stability vs performance: FS ROI vs best CNN per scaler',
             fontsize=12, fontweight='bold', y=1.01)
plt.savefig(root_dir / 'brainage_agg/outputs/figures/numerical_stability.png',
            dpi=150, bbox_inches='tight')
plt.show()

# Note: annualized_rate rows appear as NaN until results.csv is regenerated with
# the new Arm B target (sbatch brainage_agg/slurm/submit_run.sh).

# ── Summary ──────────────────────────────────────────────────────────────────
print('\nCancellation × MAE summary:')
for _, m in model_df.iterrows():
    sub = mae_df[mae_df['model']==m['model']]
    print(f"  {m['model']:20s}  canc={m['cancellation']:.4f}  "
          f"mean_MAE_change={sub[sub['aggregation'].isin(AGG_CHANGE)]['MAE'].mean():.4f}  "
          f"mean_MAE_state={sub[~sub['aggregation'].isin(AGG_CHANGE)]['MAE'].mean():.4f}")


### 18a  Cancellation score distributions — change aggregations only

Cancellation only applies to aggregations that compute a **subtraction**:
`difference`, `annualized_rate`, and `lme_slope_change` / `lme_slope`.

`mean` and `concatenation` are **immune**: they never subtract two large
nearly-equal quantities, so floating-point cancellation cannot occur.
They are excluded from this distribution and will appear as MAE reference
lines in §18b instead.


In [ ]:
from brainage_agg.features.loader import load_features, align_to_manifest as _align

if '_sd_canc' not in dir():
    _Xfs, _mfs, _ = load_features(_fs_npz)
    _sd_canc = _align(_Xfs, _mfs, _manifest_f)

# Only aggregations that involve subtraction — cancellation is meaningful here
_CHANGE_AGG_COHORT = {
    'annualized_rate':  ('cohort_all', True),
    'lme_slope':        ('cohort_all', False),   # Arm A
    'difference':       ('cohort_all', True),
    'lme_slope_change': ('cohort_all', True),    # Arm B
}
_AGG_COLORS_C = {
    'annualized_rate':  '#FB8C00',
    'lme_slope':        '#7B1FA2',
    'difference':       '#1E88E5',
    'lme_slope_change': '#AB47BC',
}

def _subj_canc_scores(agg):
    cohort_col, excl_b = _CHANGE_AGG_COHORT[agg]
    mdf = _manifest_f[_manifest_f[cohort_col]].copy()
    if excl_b: mdf = mdf[~mdf['exclude_arm_b']]
    vals = []
    for _, row in mdf.iterrows():
        sid = row['subject_id']
        if sid not in _sd_canc: continue
        feats = _sd_canc[sid]['features'].astype(np.float64)
        diff  = np.abs(feats[-1] - feats[0])
        denom = np.maximum(np.maximum(np.abs(feats[-1]), np.abs(feats[0])), 1e-6)
        vals.append(float(np.median(diff / denom)))
    return vals

_per_agg_canc = {agg: _subj_canc_scores(agg) for agg in _CHANGE_AGG_COHORT}

_ARM_A_CHANGE = ['annualized_rate', 'lme_slope']
_ARM_B_CHANGE = ['difference', 'lme_slope_change']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, agg_list, title in [
    (axes[0], _ARM_A_CHANGE, 'Arm A — change aggregations'),
    (axes[1], _ARM_B_CHANGE, 'Arm B — change aggregations'),
]:
    long_rows = []
    for agg in agg_list:
        for v in _per_agg_canc[agg]:
            long_rows.append({'aggregation': AGG_LABELS.get(agg, agg), 'cancellation': v})
    long = pd.DataFrame(long_rows)
    palette = {AGG_LABELS.get(a, a): _AGG_COLORS_C[a] for a in agg_list}

    sns.histplot(
        data=long, x='cancellation', hue='aggregation',
        multiple='dodge', bins=12, stat='density',
        palette=palette, alpha=0.85, edgecolor='white', linewidth=0.4,
        ax=ax, hue_order=[AGG_LABELS.get(a, a) for a in agg_list],
    )
    for agg in agg_list:
        ax.axvline(np.mean(_per_agg_canc[agg]),
                   color=_AGG_COLORS_C[agg], lw=1.8, linestyle='--', alpha=0.85)

    leg = ax.get_legend()
    if leg is not None:
        leg.set_title('Aggregation')
        plt.setp(leg.get_texts(), fontsize=9)
        plt.setp(leg.get_title(), fontsize=9)

    ax.set(xlabel='Per-subject cancellation score\n(higher = more relative change → less cancellation risk)',
           ylabel='Density', title=title)

plt.suptitle('Cancellation score distributions (change aggregations only — FS ROI features)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(root_dir / 'brainage_agg/outputs/figures/canc_distribution_by_agg.png',
            dpi=150, bbox_inches='tight')
plt.show()

for agg in _ARM_A_CHANGE + _ARM_B_CHANGE:
    v = _per_agg_canc[agg]
    print(f"  {agg:22s}  n={len(v):3d}  mean={np.mean(v):.4f}  std={np.std(v):.4f}")


### 18b  Cancellation score vs MAE — FS ROI (left) and best CNN per scaler (right)

Each scatter point = one **(subject, change-aggregation)** pair.
x = per-subject cancellation score, y = per-subject absolute error (after pipeline rerun)
or fold MAE (fallback until rerun).

**`mean` and `concatenation`** are shown as **horizontal dashed reference lines** (no x
value — cancellation does not apply to them) so their MAE can still be compared.


In [ ]:
_ARM_A_CHANGE_S = ['annualized_rate', 'lme_slope']
_ARM_B_CHANGE_S = ['difference', 'lme_slope_change', 'concatenation']
_STATE_AGGS     = ['mean', 'concatenation']
_STATE_COLORS   = {'mean': '#78909C', 'concatenation': '#B0BEC5'}
_STATE_STYLE    = {'mean': '--', 'concatenation': ':'}

_MARKERS = {
    'annualized_rate': '^', 'lme_slope': 'D',
    'difference': 'v', 'lme_slope_change': '<',
    'mean': 's', 'concatenation': 'P',
}

# Add state aggs for Arm A scatter; cohort_concat applies to concatenation
_CHANGE_AGG_COHORT = {**_CHANGE_AGG_COHORT,
                      'mean':          ('cohort_all',    False),
                      'concatenation': ('cohort_concat', True)}
_AGG_COLORS_ALL = {**_AGG_COLORS_C, **_STATE_COLORS}

_has_per_subject = ('per_subject' in df.columns and
                    df['per_subject'].notna().any() and
                    df['per_subject'].iloc[0] not in ('[]', '', None))

def _scatter_data_change(arm, change_aggs):
    rows = []
    for _, m in model_df.iterrows():
        # Build per-subject cancellation lookup for this extractor's features
        from brainage_agg.features.loader import load_features, align_to_manifest as _align2
        _Xtmp, _mtmp, _ = load_features(
            _feat_dir / f"{m['extractor']}.npz"
        )
        _sd_tmp = _align2(_Xtmp, _mtmp, _manifest_f)

        for agg in change_aggs:
            cohort_col, excl_b = _CHANGE_AGG_COHORT[agg]
            mdf_sub = _manifest_f[_manifest_f[cohort_col]].copy()
            if excl_b: mdf_sub = mdf_sub[~mdf_sub['exclude_arm_b']]

            cell_df = df[(df['target_arm']==arm) & (df['aggregation']==agg) &
                         (df['age_band']=='all') & (df['extractor']==m['extractor'])]
            if cell_df.empty: continue

            if _has_per_subject:
                for _, fold_row in cell_df.iterrows():
                    for subj in json.loads(fold_row['per_subject']):
                        sid = subj['subject_id']
                        if sid not in _sd_tmp: continue
                        feats = _sd_tmp[sid]['features'].astype(np.float64)
                        diff  = np.abs(feats[-1] - feats[0])
                        denom = np.maximum(np.maximum(np.abs(feats[-1]), np.abs(feats[0])), 1e-6)
                        rows.append({'model': m['model'], 'ext_type': m['extractor_type'],
                                     'scaler': m['scaler'],
                                     'cancellation': float(np.median(diff / denom)),
                                     'aggregation': agg, 'MAE': subj['abs_error']})
            else:
                for _, fold_row in cell_df.iterrows():
                    rows.append({'model': m['model'], 'ext_type': m['extractor_type'],
                                 'scaler': m['scaler'],
                                 'cancellation': m['cancellation'],
                                 'aggregation': agg, 'MAE': fold_row['MAE']})
    return pd.DataFrame(rows)

def _state_mae(arm, ext_type, agg):
    sub = df[(df['target_arm']==arm) & (df['aggregation']==agg) &
             (df['age_band']=='all') &
             (df['extractor_type']==ext_type)]
    if ext_type == 'cnn':
        # Use best CNN per scaler mean
        sub = sub[sub['extractor'].isin(model_df[model_df['ext_type']=='cnn']['extractor'].values
                                         if 'ext_type' in model_df.columns
                                         else model_df[model_df['extractor_type']=='cnn']['extractor'].values
                                         if 'extractor_type' in model_df.columns else [])]
    return sub['MAE'].mean() if len(sub) else np.nan

if not _has_per_subject:
    print("Fallback: using fold-level MAE (5 pts/agg). Rerun pipeline for per-subject points.")

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle(
    'Cancellation score vs prediction error — change aggs only\n'
    + ('(per-subject)' if _has_per_subject else '(fold-level fallback — rerun for per-subject)')
    + '  |  * p<0.05  ** p<0.01  *** p<0.001  (OLS slope test)',
    fontsize=13, fontweight='bold', y=1.01,
)

for row_idx, (arm, change_aggs, arm_label) in enumerate([
    ('A', _ARM_A_CHANGE_S + _STATE_AGGS, 'Arm A'),
    ('B', _ARM_B_CHANGE_S, 'Arm B'),
]):
    sdf = _scatter_data_change(arm, change_aggs)
    if sdf.empty: continue
    y_vals = sdf['MAE'].dropna().values
    state_maes = {agg: _state_mae(arm, 'fs_roi', agg) for agg in _STATE_AGGS}
    state_maes_cnn = {agg: _state_mae(arm, 'cnn', agg) for agg in _STATE_AGGS}
    all_y = np.concatenate([y_vals,
                             [v for v in list(state_maes.values()) + list(state_maes_cnn.values())
                              if np.isfinite(v)]])
    ymin = -5 if arm == 'A' else -0.5
    ymax = np.nanpercentile(all_y, 99) * 1.05

    for col_idx, (ext_type, panel_title, s_maes) in enumerate([
        ('fs_roi', 'FS ROI',          state_maes),
        ('cnn',    'CNN best/scaler',  state_maes_cnn),
    ]):
        ax = axes[row_idx][col_idx]
        sub = sdf[sdf['ext_type']==ext_type]
        legend_handles = []

        # ── scatter points for change aggregations ──
        for agg in change_aggs:
            pts = sub[sub['aggregation']==agg].dropna(subset=['cancellation','MAE'])
            if pts.empty: continue
            c, mk = _AGG_COLORS_ALL[agg], _MARKERS[agg]
            ax.scatter(pts['cancellation'], pts['MAE'],
                       color=c, marker=mk, s=20, alpha=0.55, zorder=3, edgecolors='none')
            _sig = ''
            if len(pts) >= 5 and pts['cancellation'].nunique() > 1:
                slope, intercept, _, p_val, _ = stats.linregress(pts['cancellation'], pts['MAE'])
                xr = np.linspace(pts['cancellation'].min(), pts['cancellation'].max(), 50)
                ax.plot(xr, slope*xr + intercept, color=c, lw=2, alpha=0.9)
                if   p_val < 0.001: _sig = '***'
                elif p_val < 0.01:  _sig = '**'
                elif p_val < 0.05:  _sig = '*'
                if _sig:
                    ax.annotate(_sig, xy=(xr[-1], slope*xr[-1]+intercept),
                                xytext=(4, 0), textcoords='offset points',
                                color=c, fontsize=12, fontweight='bold', va='center')
            import matplotlib.lines as mlines
            _lbl = AGG_LABELS.get(agg, agg) + (f'  {_sig}' if _sig else '')
            legend_handles.append(mlines.Line2D([], [], color=c, marker=mk,
                                                markersize=7, linestyle='-',
                                                label=_lbl))

        # ── horizontal reference lines for state aggregations ──
        for agg in _STATE_AGGS:
            if agg in change_aggs: continue  # already plotted as scatter
            mae_ref = s_maes.get(agg, np.nan)
            if np.isnan(mae_ref): continue
            c = _STATE_COLORS[agg]
            ls = _STATE_STYLE[agg]
            ax.axhline(mae_ref, color=c, linestyle=ls, lw=1.8, alpha=0.9, zorder=2)
            legend_handles.append(mlines.Line2D([], [], color=c, linestyle=ls, lw=2,
                                                label=f"{AGG_LABELS.get(agg,agg)} MAE (ref)"))

        ax.set(xlabel='Per-subject cancellation score',
               ylabel='Absolute error' if _has_per_subject else 'Fold MAE',
               title=f'{arm_label} — {panel_title}',
               ylim=(ymin, ymax))
        ax.legend(handles=legend_handles, fontsize=8, loc='upper right')

plt.tight_layout()
plt.savefig(root_dir / 'brainage_agg/outputs/figures/canc_scatter_roi_vs_cnn.png',
            dpi=150, bbox_inches='tight')
plt.show()
